# Phase 2 — Data Understanding and Preparation

**Notebook scope:** this notebook starts after the initial research and base synthetic data generation.  
It focuses on validating the generated datasets, cleaning/enriching them, integrating data, calculating KPIs, and preparing JSON payloads for the report-generation agents.


## 2.3 Synthetic Data Validation

Validate the generated CSV datasets by checking table loading, key relationships, foreign keys, referential integrity, and calculation consistency.


In [1]:
import pandas as pd
import os

# Load all tables
tables = {}
path = "gen_data/"
for f in os.listdir(path):
    if f.endswith(".csv"):
        name = f.replace(".csv", "")
        tables[name] = pd.read_csv(path + f)

# 1. FK integrity — the most important check
def check_fk(child_df, parent_df, fk_col, pk_col, child_name, parent_name):
    orphans = ~child_df[fk_col].isin(parent_df[pk_col])
    if orphans.sum() > 0:
        print(f"FAIL {child_name}.{fk_col} -> {parent_name}: {orphans.sum()} orphan rows")
    else:
        print(f"OK   {child_name}.{fk_col} -> {parent_name}")

banks = tables["banks"]
check_fk(tables["exposures"], banks, "bank_id", "bank_id", "exposures", "banks")
check_fk(tables["counterparty_emissions"], tables["counterparties"], 
         "counterparty_id", "counterparty_id", "counterparty_emissions", "counterparties")
check_fk(tables["utility_invoices"], tables["facilities"], 
         "facility_id", "facility_id", "utility_invoices", "facilities")
# repeat for all FK links in your data dictionary

OK   exposures.bank_id -> banks
OK   counterparty_emissions.counterparty_id -> counterparties
OK   utility_invoices.facility_id -> facilities


In [2]:
# === BANK_ID links (all tables -> banks) ===
for table_name in [
    "board_minutes_extract", "carbon_credits", "climate_risk_register",
    "climate_scenarios", "collateral", "counterparties", "employees",
    "exposures", "facilities", "financial_summary", "governance",
    "internal_carbon_price", "investments", "physical_risk_exposures",
    "rec_registry", "targets", "travel_records", "utility_invoices",
    "value_chain_map", "vehicles"
]:
    check_fk(tables[table_name], banks, "bank_id", "bank_id", table_name, "banks")

# === Cross-table FK links ===
check_fk(tables["collateral"], tables["exposures"],
         "exposure_id", "exposure_id", "collateral", "exposures")

check_fk(tables["collateral"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "collateral", "counterparties")

check_fk(tables["counterparty_emissions"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "counterparty_emissions", "counterparties")

check_fk(tables["exposures"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "exposures", "counterparties")

check_fk(tables["investments"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "investments", "counterparties")

check_fk(tables["physical_risk_exposures"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "physical_risk_exposures", "counterparties")

check_fk(tables["utility_invoices"], tables["facilities"],
         "facility_id", "facility_id", "utility_invoices", "facilities")

check_fk(tables["disclosure_data_map"], tables["disclosures"],
         "disclosure_id", "disclosure_id", "disclosure_data_map", "disclosures")

check_fk(tables["disclosure_data_map"], tables["data_requirements"],
         "data_req_id", "data_req_id", "disclosure_data_map", "data_requirements")# === BANK_ID links (all tables -> banks) ===
for table_name in [
    "board_minutes_extract", "carbon_credits", "climate_risk_register",
    "climate_scenarios", "collateral", "counterparties", "employees",
    "exposures", "facilities", "financial_summary", "governance",
    "internal_carbon_price", "investments", "physical_risk_exposures",
    "rec_registry", "targets", "travel_records", "utility_invoices",
    "value_chain_map", "vehicles"
]:
    check_fk(tables[table_name], banks, "bank_id", "bank_id", table_name, "banks")

# === Cross-table FK links ===
check_fk(tables["collateral"], tables["exposures"],
         "exposure_id", "exposure_id", "collateral", "exposures")

check_fk(tables["collateral"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "collateral", "counterparties")

check_fk(tables["counterparty_emissions"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "counterparty_emissions", "counterparties")

check_fk(tables["exposures"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "exposures", "counterparties")

check_fk(tables["investments"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "investments", "counterparties")

check_fk(tables["physical_risk_exposures"], tables["counterparties"],
         "counterparty_id", "counterparty_id", "physical_risk_exposures", "counterparties")

check_fk(tables["utility_invoices"], tables["facilities"],
         "facility_id", "facility_id", "utility_invoices", "facilities")

check_fk(tables["disclosure_data_map"], tables["disclosures"],
         "disclosure_id", "disclosure_id", "disclosure_data_map", "disclosures")

check_fk(tables["disclosure_data_map"], tables["data_requirements"],
         "data_req_id", "data_req_id", "disclosure_data_map", "data_requirements")

OK   board_minutes_extract.bank_id -> banks
OK   carbon_credits.bank_id -> banks
OK   climate_risk_register.bank_id -> banks
OK   climate_scenarios.bank_id -> banks
OK   collateral.bank_id -> banks
OK   counterparties.bank_id -> banks
OK   employees.bank_id -> banks
OK   exposures.bank_id -> banks
OK   facilities.bank_id -> banks
OK   financial_summary.bank_id -> banks
OK   governance.bank_id -> banks
OK   internal_carbon_price.bank_id -> banks
OK   investments.bank_id -> banks
OK   physical_risk_exposures.bank_id -> banks
OK   rec_registry.bank_id -> banks
OK   targets.bank_id -> banks
OK   travel_records.bank_id -> banks
OK   utility_invoices.bank_id -> banks
OK   value_chain_map.bank_id -> banks
OK   vehicles.bank_id -> banks
OK   collateral.exposure_id -> exposures
OK   collateral.counterparty_id -> counterparties
OK   counterparty_emissions.counterparty_id -> counterparties
OK   exposures.counterparty_id -> counterparties
FAIL investments.counterparty_id -> counterparties: 185 orp

In [3]:
inv = tables["investments"]
cp = tables["counterparties"]

print("investments rows:", len(inv))
print("matched:", inv["counterparty_id"].isin(cp["counterparty_id"]).sum())
print("\nSample investment counterparty_ids:", inv["counterparty_id"].unique()[:5])
print("Sample counterparties counterparty_ids:", cp["counterparty_id"].unique()[:5])

# Travel emissions check
travel = tables["travel_records"]
travel["emissions_check"] = travel["distance_km"] * travel["emission_factor_kg_co2e_per_km"]
travel["emissions_delta_pct"] = (
    (travel["emissions_kg_co2e"] - travel["emissions_check"]) / travel["emissions_check"] * 100
).abs()
print("=== TRAVEL ===")
print("Rows with >1% delta:", (travel["emissions_delta_pct"] > 1).sum())
print("Max delta %:", travel["emissions_delta_pct"].max().round(4))

# Scope 2 location check
util = tables["utility_invoices"]
util["scope2_loc_check"] = util["electricity_kwh"] * util["grid_emission_factor_tco2e_per_mwh"] / 1000
util["scope2_delta_pct"] = (
    (util["scope2_location_tco2e"] - util["scope2_loc_check"]) / util["scope2_loc_check"] * 100
).abs()
print("\n=== SCOPE 2 LOCATION ===")
print("Rows with >1% delta:", (util["scope2_delta_pct"] > 1).sum())
print("Max delta %:", util["scope2_delta_pct"].max().round(4))

# Vehicle Scope 1 check
veh = tables["vehicles"]
veh["scope1_check"] = veh["annual_km_2024"] * veh["emissions_g_co2_per_km"] / 1_000_000
veh["scope1_delta_pct"] = (
    (veh["scope1_tco2e_2024"] - veh["scope1_check"]) / veh["scope1_check"].replace(0, float("nan")) * 100
).abs()
print("\n=== VEHICLES SCOPE 1 ===")
print("Rows with >1% delta:", (veh["scope1_delta_pct"] > 1).sum())
print("Max delta %:", veh["scope1_delta_pct"].max().round(4))

investments rows: 185
matched: 0

Sample investment counterparty_ids: [nan]
Sample counterparties counterparty_ids: <StringArray>
['CP00001', 'CP00002', 'CP00003', 'CP00004', 'CP00005']
Length: 5, dtype: str
=== TRAVEL ===
Rows with >1% delta: 1
Max delta %: 1.0584

=== SCOPE 2 LOCATION ===
Rows with >1% delta: 0
Max delta %: 0.0185

=== VEHICLES SCOPE 1 ===
Rows with >1% delta: 379
Max delta %: 172.6145


In [4]:
print(tables["investments"].columns.tolist())
print(tables["investments"].head(3).to_string())

veh = tables["vehicles"]

# The formula: km * g_co2_per_km / 1,000,000 = tco2e
# BUT electric vehicles (emissions_g_co2_per_km = 0) should be 0
# AND fuel consumption path may differ — check both

# First understand the pattern of failures
veh["scope1_recalc"] = veh["annual_km_2024"] * veh["emissions_g_co2_per_km"] / 1_000_000
veh["delta_pct"] = (
    (veh["scope1_tco2e_2024"] - veh["scope1_recalc"]) / 
    veh["scope1_recalc"].replace(0, float("nan")) * 100
).abs()

# Break down failures by fuel type
print("=== DELTA BY FUEL TYPE ===")
print(veh.groupby("fuel_type")["delta_pct"].agg(["count","mean","max"]).round(2))

# Check if fuel consumption path explains it better
# natural gas: consumption_l * 2.68 kg_co2/l / 1000 = tco2e (DEFRA factor)
# petrol: consumption_l * 2.31 kg_co2/l / 1000 = tco2e
veh["scope1_fuel_path"] = veh.apply(lambda r: 
    r["annual_fuel_consumption_l"] * 2.31 / 1000 if r["fuel_type"] == "petrol"
    else r["annual_fuel_consumption_l"] * 2.68 / 1000 if r["fuel_type"] == "hybrid"
    else 0,  # electric
    axis=1
)
veh["delta_fuel_path"] = (
    (veh["scope1_tco2e_2024"] - veh["scope1_fuel_path"]) / 
    veh["scope1_fuel_path"].replace(0, float("nan")) * 100
).abs()

print("\n=== FUEL PATH DELTA BY FUEL TYPE ===")
print(veh.groupby("fuel_type")["delta_fuel_path"].agg(["count","mean","max"]).round(2))


veh = tables["vehicles"]

print("=== EMISSION FACTOR SOURCE DISTRIBUTION ===")
print(veh["emission_factor_source"].value_counts())

print("\n=== DELTA BY EMISSION FACTOR SOURCE ===")
veh["scope1_recalc"] = veh["annual_km_2024"] * veh["emissions_g_co2_per_km"] / 1_000_000
veh["delta_pct"] = (
    (veh["scope1_tco2e_2024"] - veh["scope1_recalc"]) /
    veh["scope1_recalc"].replace(0, float("nan")) * 100
).abs()
print(veh.groupby("emission_factor_source")["delta_pct"].agg(["count","mean","max"]).round(2))

print("\n=== DELTA BY FUEL TYPE ===")
print(veh.groupby("fuel_type")["delta_pct"].agg(["count","mean","max"]).round(2))

# Check electrics specifically — should be 0 scope1
print("\n=== ELECTRIC VEHICLES scope1 check ===")
ev = veh[veh["fuel_type"] == "electric"]
print("EV rows:", len(ev))
print("EV scope1_tco2e_2024 non-zero:", (ev["scope1_tco2e_2024"] > 0).sum())
print("EV emissions_g_co2_per_km non-zero:", (ev["emissions_g_co2_per_km"] > 0).sum())

['investment_id', 'bank_id', 'asset_class', 'issuer_name', 'nace_code', 'country', 'nominal_amount_meur', 'market_value_meur', 'currency', 'esg_classification', 'purchase_date', 'issuer_revenue_meur', 'issuer_evic_meur', 'ppp_gdp_meur', 'pcaf_data_quality_score', 'counterparty_id', 'reporting_year']
  investment_id bank_id     asset_class    issuer_name nace_code country  nominal_amount_meur  market_value_meur currency esg_classification purchase_date  issuer_revenue_meur  issuer_evic_meur  ppp_gdp_meur  pcaf_data_quality_score  counterparty_id  reporting_year
0      INV00001  BANK01   listed_equity  Manufacture 1       C25      IT                27.98              23.93      EUR          article_8    2019-05-26                168.8             457.8           NaN                        2              NaN            2024
1      INV00002  BANK01  sovereign_bond    Sovereign 2       O84      DE                78.18              83.78      EUR          article_6    2019-12-11             

In [5]:
veh = tables["vehicles"]

# For each non-electric fuel type, back-calculate what factor was actually used
# formula: scope1_tco2e = consumption_l * factor / 1000
# therefore: factor = scope1_tco2e * 1000 / consumption_l

non_ev = veh[
    (veh["fuel_type"] != "electric") & 
    (veh["annual_fuel_consumption_l"] > 0) &
    (veh["scope1_tco2e_2024"] > 0)
].copy()

non_ev["implied_factor"] = (
    non_ev["scope1_tco2e_2024"] * 1000 / non_ev["annual_fuel_consumption_l"]
)

print("=== IMPLIED EMISSION FACTORS BY FUEL TYPE ===")
print(non_ev.groupby("fuel_type")["implied_factor"].agg(["mean","std","min","max"]).round(4))

=== IMPLIED EMISSION FACTORS BY FUEL TYPE ===
           mean  std     min     max
fuel_type                           
diesel     2.67  0.0  2.6699  2.6701
hybrid     2.67  0.0  2.6699  2.6701
petrol     2.67  0.0  2.6699  2.6701


## 2.4 Data Cleaning and Standardization

Apply corrections and standardization needed before integration and export.  
This includes vehicle emissions correction, IFRS disclosure catalog expansion, climate opportunity enrichment, and scenario methodology/resilience fields.


In [6]:
veh = tables["vehicles"].copy()

EMISSION_FACTOR = 2.67  # kg CO2 per litre, consistent across all fuel types

def recalc_scope1(row):
    if row["fuel_type"] == "electric":
        return 0.0  # EVs have zero Scope 1 by definition
    if row["annual_fuel_consumption_l"] <= 0:
        return 0.0
    return round(row["annual_fuel_consumption_l"] * EMISSION_FACTOR / 1000, 6)

veh["scope1_tco2e_2024_clean"] = veh.apply(recalc_scope1, axis=1)

# Verify the fix
veh["delta_pct"] = (
    (veh["scope1_tco2e_2024_clean"] - veh["scope1_tco2e_2024"]) /
    veh["scope1_tco2e_2024"].replace(0, float("nan")) * 100
).abs()

print("=== VERIFICATION ===")
print(f"EV rows forced to 0: {(veh['fuel_type'] == 'electric').sum()}")
print(f"Rows still >1% delta: {(veh['delta_pct'] > 1).sum()}")
print(f"Max delta %: {veh['delta_pct'].max().round(4)}")

print("\n=== SCOPE 1 TOTALS COMPARISON (per bank, 2024) ===")
comparison = veh.groupby("bank_id").agg(
    original=("scope1_tco2e_2024", "sum"),
    recalculated=("scope1_tco2e_2024_clean", "sum")
).round(2)
comparison["diff"] = (comparison["recalculated"] - comparison["original"]).round(2)
print(comparison)

# Save clean version
veh.to_csv("gen_data/vehicles.csv", index=False)
print("\nvehicles_clean.csv saved")

=== VERIFICATION ===
EV rows forced to 0: 62
Rows still >1% delta: 53
Max delta %: 100.0

=== SCOPE 1 TOTALS COMPARISON (per bank, 2024) ===
         original  recalculated  diff
bank_id                              
BANK01     623.72        622.58 -1.14
BANK02     476.17        475.52 -0.65
BANK03     276.17        276.16 -0.01
BANK04      28.41         27.25 -1.16
BANK05     146.50        145.99 -0.51

vehicles_clean.csv saved


In [7]:
# Confirm the 53 remaining delta rows are all EVs
still_delta = veh[(veh["delta_pct"] > 1)]
print("Fuel types of remaining delta rows:")
print(still_delta["fuel_type"].value_counts())
# Should show: electric    53

Fuel types of remaining delta rows:
fuel_type
electric    53
Name: count, dtype: int64


In [8]:
# 2.4.1 Expand disclosures.csv to full IFRS S1/S2 catalog
# Replaces the original 17-row partial catalog with a 44-row full coverage catalog.
# This is the knowledge base for the IFRS Compliance Critic Agent.

import pandas as pd

disclosures_full = pd.DataFrame([
    # ── IFRS S1 — Governance ──────────────────────────────────────────────
    {"disclosure_id": "S1.27",  "standard": "IFRS_S1", "topic": "Governance",
     "paragraph": "27", "disclosure_type": "narrative",
     "title": "Governance body oversight of sustainability risks and opportunities"},
    {"disclosure_id": "S1.28",  "standard": "IFRS_S1", "topic": "Governance",
     "paragraph": "28", "disclosure_type": "narrative",
     "title": "Governance body skills and competencies for sustainability oversight"},
    {"disclosure_id": "S1.29",  "standard": "IFRS_S1", "topic": "Governance",
     "paragraph": "29", "disclosure_type": "narrative",
     "title": "Management role in sustainability oversight"},

    # ── IFRS S1 — Strategy ────────────────────────────────────────────────
    {"disclosure_id": "S1.33",  "standard": "IFRS_S1", "topic": "Strategy",
     "paragraph": "33", "disclosure_type": "narrative",
     "title": "Sustainability-related risks and opportunities affecting the entity"},
    {"disclosure_id": "S1.34",  "standard": "IFRS_S1", "topic": "Strategy",
     "paragraph": "34", "disclosure_type": "narrative",
     "title": "Effects on business model and value chain"},
    {"disclosure_id": "S1.35",  "standard": "IFRS_S1", "topic": "Strategy",
     "paragraph": "35", "disclosure_type": "narrative",
     "title": "Effects on strategy and decision-making"},
    {"disclosure_id": "S1.36",  "standard": "IFRS_S1", "topic": "Strategy",
     "paragraph": "36", "disclosure_type": "narrative",
     "title": "Effects on financial position, performance and cash flows"},
    {"disclosure_id": "S1.37",  "standard": "IFRS_S1", "topic": "Strategy",
     "paragraph": "37", "disclosure_type": "narrative",
     "title": "Resilience of strategy and business model to sustainability risks"},

    # ── IFRS S1 — Risk Management ─────────────────────────────────────────
    {"disclosure_id": "S1.41",  "standard": "IFRS_S1", "topic": "Risk_management",
     "paragraph": "41", "disclosure_type": "narrative",
     "title": "Processes to identify and assess sustainability-related risks"},
    {"disclosure_id": "S1.42",  "standard": "IFRS_S1", "topic": "Risk_management",
     "paragraph": "42", "disclosure_type": "narrative",
     "title": "Processes to manage sustainability-related risks"},
    {"disclosure_id": "S1.43",  "standard": "IFRS_S1", "topic": "Risk_management",
     "paragraph": "43", "disclosure_type": "narrative",
     "title": "Integration of risk identification and management into overall risk framework"},

    # ── IFRS S1 — Metrics and Targets ─────────────────────────────────────
    {"disclosure_id": "S1.50",  "standard": "IFRS_S1", "topic": "Metrics_targets",
     "paragraph": "50", "disclosure_type": "quantitative",
     "title": "Metrics used to measure and monitor sustainability-related risks"},
    {"disclosure_id": "S1.51",  "standard": "IFRS_S1", "topic": "Metrics_targets",
     "paragraph": "51", "disclosure_type": "quantitative",
     "title": "Cross-industry metric categories"},
    {"disclosure_id": "S1.52",  "standard": "IFRS_S1", "topic": "Metrics_targets",
     "paragraph": "52", "disclosure_type": "narrative",
     "title": "Targets set to manage sustainability-related risks and opportunities"},

    # ── IFRS S2 — Governance ──────────────────────────────────────────────
    {"disclosure_id": "S2.6a",  "standard": "IFRS_S2", "topic": "Governance",
     "paragraph": "6(a)", "disclosure_type": "narrative",
     "title": "Board oversight of climate-related risks and opportunities"},
    {"disclosure_id": "S2.6a_iii", "standard": "IFRS_S2", "topic": "Governance",
     "paragraph": "6(a)(iii)", "disclosure_type": "narrative",
     "title": "Board oversight: how board is informed about climate risks"},
    {"disclosure_id": "S2.6a_v",  "standard": "IFRS_S2", "topic": "Governance",
     "paragraph": "6(a)(v)", "disclosure_type": "narrative",
     "title": "Board oversight: how performance targets for climate are monitored"},
    {"disclosure_id": "S2.6b",  "standard": "IFRS_S2", "topic": "Governance",
     "paragraph": "6(b)", "disclosure_type": "narrative",
     "title": "Management role in climate risk assessment and management"},

    # ── IFRS S2 — Strategy ────────────────────────────────────────────────
    {"disclosure_id": "S2.9",   "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "9", "disclosure_type": "narrative",
     "title": "Climate-related risks and opportunities identified over short, medium and long term"},
    {"disclosure_id": "S2.10",  "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "10", "disclosure_type": "narrative",
     "title": "Definition of short, medium and long term time horizons"},
    {"disclosure_id": "S2.11",  "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "11", "disclosure_type": "narrative",
     "title": "Concentration of transition and physical risks in value chain"},
    {"disclosure_id": "S2.13",  "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "13", "disclosure_type": "narrative",
     "title": "Effects of climate risks and opportunities on business model and value chain"},
    {"disclosure_id": "S2.15",  "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "15", "disclosure_type": "narrative",
     "title": "Current and anticipated financial effects of climate risks and opportunities"},
    {"disclosure_id": "S2.19",  "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "19", "disclosure_type": "narrative",
     "title": "Climate-related transition plan"},
    {"disclosure_id": "S2.22",  "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "22", "disclosure_type": "narrative",
     "title": "Climate scenario analysis inputs, assumptions, and analytical choices"},
    {"disclosure_id": "S2.22b", "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "22(b)", "disclosure_type": "narrative",
     "title": "Scenario assumptions: carbon price, GDP growth, renewable share, technology readiness"},
    {"disclosure_id": "S2.29e", "standard": "IFRS_S2", "topic": "Strategy",
     "paragraph": "29(e)", "disclosure_type": "narrative",
     "title": "Resilience of strategy assessed through scenario analysis"},

    # ── IFRS S2 — Risk Management ─────────────────────────────────────────
    {"disclosure_id": "S2.25a", "standard": "IFRS_S2", "topic": "Risk_management",
     "paragraph": "25(a)", "disclosure_type": "narrative",
     "title": "Processes to identify and assess climate-related risks"},
    {"disclosure_id": "S2.25b", "standard": "IFRS_S2", "topic": "Risk_management",
     "paragraph": "25(b)", "disclosure_type": "narrative",
     "title": "Inputs and parameters used in risk assessment including data sources"},
    {"disclosure_id": "S2.25c", "standard": "IFRS_S2", "topic": "Risk_management",
     "paragraph": "25(c)", "disclosure_type": "narrative",
     "title": "Whether and how transition and physical risks are prioritised"},
    {"disclosure_id": "S2.31",  "standard": "IFRS_S2", "topic": "Risk_management",
     "paragraph": "31", "disclosure_type": "narrative",
     "title": "Integration of climate risk identification into overall enterprise risk management"},

    # ── IFRS S2 — Metrics and Targets ─────────────────────────────────────
    {"disclosure_id": "S2.29a_i",   "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "29(a)(i)", "disclosure_type": "quantitative",
     "title": "Absolute gross Scope 1 GHG emissions"},
    {"disclosure_id": "S2.29a_ii",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "29(a)(ii)", "disclosure_type": "quantitative",
     "title": "Absolute gross Scope 2 GHG emissions (location-based and market-based)"},
    {"disclosure_id": "S2.29a_iii", "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "29(a)(iii)", "disclosure_type": "quantitative",
     "title": "Absolute gross Scope 3 GHG emissions and categories included"},
    {"disclosure_id": "S2.29b",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "29(b)", "disclosure_type": "quantitative",
     "title": "Transition and physical climate risk exposure metrics"},
    {"disclosure_id": "S2.29f",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "29(f)", "disclosure_type": "quantitative",
     "title": "Internal carbon price applied to decision-making"},
    {"disclosure_id": "S2.29g",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "29(g)", "disclosure_type": "quantitative",
     "title": "Executive remuneration linked to climate-related considerations"},
    {"disclosure_id": "S2.33",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "33", "disclosure_type": "narrative",
     "title": "Targets used to manage climate-related risks and opportunities"},
    {"disclosure_id": "S2.34",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "34", "disclosure_type": "quantitative",
     "title": "Quantitative and qualitative targets including milestones"},
    {"disclosure_id": "S2.36e", "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "36(e)", "disclosure_type": "narrative",
     "title": "Use of carbon credits including type, quality and quantity retired"},

    # ── IFRS S2 — Industry-based (BCBS/banking) ───────────────────────────
    {"disclosure_id": "S2.B62",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "B62", "disclosure_type": "quantitative",
     "title": "Financed emissions: PCAF methodology, attribution factors, data quality scores"},
    {"disclosure_id": "S2.B62A", "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "B62A", "disclosure_type": "quantitative",
     "title": "Financed emissions from sovereign bonds using PCAF sovereign methodology"},
    {"disclosure_id": "S2.B63",  "standard": "IFRS_S2", "topic": "Metrics_targets",
     "paragraph": "B63", "disclosure_type": "quantitative",
     "title": "Portfolio carbon intensity of lending activities (tCO2e per EUR million lent)"},
])

# Validate no duplicate disclosure_ids
assert disclosures_full["disclosure_id"].duplicated().sum() == 0, \
    "Duplicate disclosure_ids found — fix before saving"

print(f"Total disclosures: {len(disclosures_full)}")
print(disclosures_full.groupby(["standard", "topic"]).size().to_string())

# Save — replaces the original 17-row file
disclosures_full.to_csv("gen_data/disclosures.csv", index=False)
print("\ngen_data/disclosures.csv saved (44 rows)")

# Also reload in tables dict so rest of notebook uses updated version
tables["disclosures"] = disclosures_full

Total disclosures: 43
standard  topic          
IFRS_S1   Governance          3
          Metrics_targets     3
          Risk_management     3
          Strategy            5
IFRS_S2   Governance          4
          Metrics_targets    12
          Risk_management     4
          Strategy            9

gen_data/disclosures.csv saved (44 rows)


In [9]:
# 2.4.2 Create climate_opportunities.csv
# Fills the IFRS S2 §9 opportunity disclosure gap.
# Each bank gets 5 opportunities reflecting its archetype.
# The Writer Agent uses this so it does NOT invent opportunity narrative.

import pandas as pd

BANK_ARCHETYPES = {
    "BANK01": "large_universal",
    "BANK02": "large_universal",
    "BANK03": "mid_size_commercial",
    "BANK04": "specialized_green",
    "BANK05": "corporate_laggard",
}

# Template: (opportunity_type, category, description, revenue_impact_meur, time_horizon, ifrs_s2_ref)
# Values are calibrated to bank size — BANK01 has 850B assets, BANK04 18B
OPPORTUNITY_TEMPLATES = {
    "large_universal": [
        ("green_loan_growth",
         "Products_and_services",
         "Expansion of green and sustainability-linked loan book aligned with EU Taxonomy, driven by corporate client transition financing demand.",
         180.0, "medium_term", "S2.9"),
        ("sustainable_bond_underwriting",
         "Products_and_services",
         "Growth in green bond and social bond underwriting and distribution fees as sovereign and corporate issuers accelerate ESG capital market activity.",
         95.0, "short_term", "S2.9"),
        ("transition_advisory",
         "Products_and_services",
         "Fee income from climate risk advisory, transition planning support, and ESG due diligence services offered to large corporate clients.",
         42.0, "medium_term", "S2.9"),
        ("renewable_energy_project_finance",
         "Products_and_services",
         "Project finance for solar, wind, and battery storage infrastructure aligned with REPowerEU and national renewable energy targets.",
         130.0, "long_term", "S2.9"),
        ("operational_energy_efficiency",
         "Resource_efficiency",
         "Reduction in energy costs from facility EPC upgrades and REC procurement, lowering Scope 2 emissions and operational expenditure.",
         18.0, "short_term", "S2.9"),
    ],
    "mid_size_commercial": [
        ("green_loan_growth",
         "Products_and_services",
         "Growth of green SME lending under LMA Green Loan Principles, targeting energy-efficient commercial real estate and manufacturing retrofits.",
         55.0, "medium_term", "S2.9"),
        ("sustainable_bond_issuance",
         "Products_and_services",
         "Self-issuance of green bonds to fund green loan portfolio, reducing funding cost and improving ESG investor access.",
         28.0, "medium_term", "S2.9"),
        ("transition_advisory",
         "Products_and_services",
         "Climate transition advisory for mid-market industrial clients subject to ETS obligations and CBAM.",
         14.0, "short_term", "S2.9"),
        ("renewable_energy_project_finance",
         "Products_and_services",
         "Project finance for regional renewable energy cooperatives and municipal solar projects.",
         40.0, "long_term", "S2.9"),
        ("operational_energy_efficiency",
         "Resource_efficiency",
         "Energy cost savings from facility LED and HVAC upgrades across branch network, supported by EPC rating improvements.",
         6.0, "short_term", "S2.9"),
    ],
    "specialized_green": [
        ("green_loan_growth",
         "Products_and_services",
         "Core business expansion: EU Taxonomy-aligned green loans now represent 47% of loan book with strong pipeline growth from SFDR Article 9 fund clients.",
         210.0, "short_term", "S2.9"),
        ("impact_investing_products",
         "Products_and_services",
         "Launch of biodiversity-linked and blue economy financing products addressing emerging EU Nature Restoration Law investment needs.",
         65.0, "medium_term", "S2.9"),
        ("transition_advisory",
         "Products_and_services",
         "Premium ESG transition advisory revenue from corporate and institutional clients benchmarking against SBTi pathways.",
         30.0, "short_term", "S2.9"),
        ("renewable_energy_project_finance",
         "Products_and_services",
         "Offshore wind and green hydrogen project finance in the Netherlands and Nordic markets, aligned with REPowerEU hydrogen strategy.",
         90.0, "long_term", "S2.9"),
        ("operational_energy_efficiency",
         "Resource_efficiency",
         "Near-zero operational emissions profile maintained through 100% renewable energy procurement; cost advantage over peers with higher Scope 2 exposure.",
         12.0, "short_term", "S2.9"),
    ],
    "corporate_laggard": [
        ("green_loan_growth",
         "Products_and_services",
         "Early-stage green loan product launch targeting Iberian SME market; green loan share currently below 2% with significant growth headroom.",
         12.0, "long_term", "S2.9"),
        ("sustainable_bond_issuance",
         "Products_and_services",
         "Planned issuance of first sustainability bond to access ESG investor base and reduce funding cost premium currently paid versus green peers.",
         8.0, "long_term", "S2.9"),
        ("transition_advisory",
         "Products_and_services",
         "Basic climate risk advisory offering under development; planned launch for large corporate segment from 2026.",
         4.0, "long_term", "S2.9"),
        ("renewable_energy_project_finance",
         "Products_and_services",
         "Participation in Iberian solar project finance syndications alongside larger arranger banks, building renewable energy expertise.",
         18.0, "long_term", "S2.9"),
        ("operational_energy_efficiency",
         "Resource_efficiency",
         "Energy audit program initiated across branch network; expected operational cost reduction upon completion of EPC improvement programme.",
         3.0, "medium_term", "S2.9"),
    ],
}

rows = []
opp_counter = 1
for bank_id, archetype in BANK_ARCHETYPES.items():
    for i, (opp_type, category, description, revenue_impact, time_horizon, ifrs_ref) in \
            enumerate(OPPORTUNITY_TEMPLATES[archetype], start=1):
        rows.append({
            "opportunity_id":              f"OPP-{bank_id}-{i:02d}",
            "bank_id":                     bank_id,
            "reporting_year":              2024,
            "opportunity_type":            opp_type,
            "category":                    category,
            "description":                 description,
            "estimated_revenue_impact_meur": revenue_impact,
            "time_horizon":                time_horizon,
            "confidence_level":            "medium" if archetype != "corporate_laggard" else "low",
            "ifrs_s2_para_evidence":       ifrs_ref,
            "data_source":                 "bank_strategy_documents",
            "linked_risk_category":        (
                "transition_policy" if "green" in opp_type or "advisory" in opp_type
                else "physical_acute" if "efficiency" in opp_type
                else "transition_market"
            ),
        })
        opp_counter += 1

climate_opportunities = pd.DataFrame(rows)

# Validation
assert len(climate_opportunities) == 25, \
    f"Expected 25 rows (5 per bank × 5 banks), got {len(climate_opportunities)}"
assert climate_opportunities["opportunity_id"].duplicated().sum() == 0, \
    "Duplicate opportunity IDs"
assert climate_opportunities["bank_id"].isin(tables["banks"]["bank_id"]).all(), \
    "Unknown bank_id in opportunities"

print(f"climate_opportunities: {len(climate_opportunities)} rows")
print(climate_opportunities.groupby(["bank_id", "opportunity_type"])["estimated_revenue_impact_meur"].first().to_string())

climate_opportunities.to_csv("gen_data/climate_opportunities.csv", index=False)
print("\ngen_data/climate_opportunities.csv saved")

# Load into tables dict
tables["climate_opportunities"] = climate_opportunities

climate_opportunities: 25 rows
bank_id  opportunity_type                
BANK01   green_loan_growth                   180.0
         operational_energy_efficiency        18.0
         renewable_energy_project_finance    130.0
         sustainable_bond_underwriting        95.0
         transition_advisory                  42.0
BANK02   green_loan_growth                   180.0
         operational_energy_efficiency        18.0
         renewable_energy_project_finance    130.0
         sustainable_bond_underwriting        95.0
         transition_advisory                  42.0
BANK03   green_loan_growth                    55.0
         operational_energy_efficiency         6.0
         renewable_energy_project_finance     40.0
         sustainable_bond_issuance            28.0
         transition_advisory                  14.0
BANK04   green_loan_growth                   210.0
         impact_investing_products            65.0
         operational_energy_efficiency        12.0
         

In [10]:
# 2.4.3 Add methodology_notes and resilience_assessment to climate_scenarios
# Required for IFRS S2 §22(b) methodology disclosure and §22(e) resilience assessment.
# Writer Agent uses these fields directly — prevents methodology hallucination.

scenarios = tables["climate_scenarios"].copy()

# ── Methodology notes per scenario name ──────────────────────────────────
# These describe how the bank applied NGFS v4 parameters to its portfolio.
# Varies by scenario type to give the Writer differentiated content.

METHODOLOGY_MAP = {
    "Net Zero 2050": (
        "NGFS v4 Net Zero 2050 scenario applied to the banking book using a top-down "
        "sector-pathway approach. Carbon price assumptions from IEA NZE 2050 trajectory "
        "mapped to counterparty NACE sectors. Stranded asset estimates derived from "
        "sector-level fossil fuel capex exposure. Analysis scope covers all lending book "
        "exposures across DE, FR, IT, NL, ES jurisdictions."
    ),
    "Below 2°C": (
        "NGFS v4 Below 2°C (orderly) scenario applied using NGFS IAM outputs aligned to "
        "a 1.8°C median temperature pathway. Carbon price trajectory from REMIND-MAgPIE "
        "model. Financial impact assessed through loan-level probability of default "
        "sensitivity to carbon cost pass-through by sector carbon intensity tier."
    ),
    "Current Policies": (
        "NGFS v4 Current Policies (hot house world) scenario used as the adverse baseline. "
        "Physical risk parameters sourced from NGFS chronic and acute hazard layers, "
        "downscaled to counterparty postcode and collateral location. RCP 8.5 physical "
        "risk projections used for 2050 horizon analysis."
    ),
    "Nationally Determined Contributions": (
        "NGFS v4 NDC scenario represents delayed but eventually strengthened policy action. "
        "Carbon pricing ramp-up assumed from 2030 onward based on Paris Agreement NDC "
        "submission trajectories. Revenue-at-risk computed from sector revenue sensitivity "
        "to carbon price at EUR/tCO2e levels specified in the scenario."
    ),
    "Delayed Transition": (
        "NGFS v4 Delayed Transition (disorderly) scenario applied to capture abrupt policy "
        "shift risk post-2030. Short-term physical risk assumed at RCP 2.6 level; "
        "transition risk shock applied as a step-change in carbon price in 2030. "
        "Particularly material for high-carbon sector exposures with medium-term loan maturities."
    ),
    "Divergent Net Zero": (
        "NGFS v4 Divergent Net Zero scenario used to capture technology pathway uncertainty. "
        "High renewable share but with divergent sectoral transition speeds across "
        "jurisdictions. Carbon price assumptions vary by country based on national ETS "
        "and carbon tax trajectories. Used primarily to stress-test technology transition "
        "risk in the automotive and energy manufacturing lending sub-portfolio."
    ),
}

# ── Resilience assessment per scenario type ───────────────────────────────
RESILIENCE_MAP = {
    "orderly": (
        "Under orderly transition scenarios, the portfolio demonstrates adequate resilience. "
        "Transition risk losses remain within Pillar 2 capital buffer thresholds. "
        "Green loan growth and client engagement on transition plans are expected to "
        "progressively reduce high-carbon sector concentration over the medium term."
    ),
    "disorderly": (
        "Under disorderly transition scenarios, near-term resilience is acceptable but "
        "medium-term capital consumption from stranded asset impairments is material. "
        "The bank's current sector exposure limits and active portfolio decarbonisation "
        "glide-path monitoring are the primary resilience mechanisms. Additional capital "
        "buffers may be required post-2030 under the most adverse disorderly assumptions."
    ),
    "hot_house": (
        "Under hot house world scenarios, physical risk losses represent the dominant "
        "long-term resilience challenge. Collateral revaluation risk in flood-exposed "
        "mortgage books and agricultural borrower default risk from chronic heat stress "
        "are identified as the primary transmission channels. The bank's flood-zone "
        "overlay and EPC-based collateral monitoring are resilience mechanisms deployed "
        "for the 2050 horizon."
    ),
}

scenarios["methodology_notes"] = scenarios["scenario_name"].map(METHODOLOGY_MAP)
scenarios["resilience_assessment"] = scenarios["scenario_type"].map(RESILIENCE_MAP)

# Validation — no nulls expected
assert scenarios["methodology_notes"].notna().all(), \
    "Missing methodology_notes for some scenarios — check METHODOLOGY_MAP keys"
assert scenarios["resilience_assessment"].notna().all(), \
    "Missing resilience_assessment for some scenarios — check RESILIENCE_MAP keys"

print(f"climate_scenarios enriched: {len(scenarios)} rows")
print("methodology_notes null:", scenarios["methodology_notes"].isna().sum())
print("resilience_assessment null:", scenarios["resilience_assessment"].isna().sum())
print("\nSample methodology_notes:")
print(scenarios[["scenario_name","scenario_type","methodology_notes"]].drop_duplicates(
    "scenario_name")[["scenario_name","methodology_notes"]].to_string())

scenarios.to_csv("gen_data/climate_scenarios.csv", index=False)
print("\ngen_data/climate_scenarios.csv saved")

tables["climate_scenarios"] = scenarios

climate_scenarios enriched: 84 rows
methodology_notes null: 0
resilience_assessment null: 0

Sample methodology_notes:
                          scenario_name                                                                                                                                                                                                                                                                                                                                                                                    methodology_notes
0                         Net Zero 2050                             NGFS v4 Net Zero 2050 scenario applied to the banking book using a top-down sector-pathway approach. Carbon price assumptions from IEA NZE 2050 trajectory mapped to counterparty NACE sectors. Stranded asset estimates derived from sector-level fossil fuel capex exposure. Analysis scope covers all lending book exposures across DE, FR, IT, NL, ES jurisdictions.
3                      

## 2.5 Data Integration and Relationship Validation

Prepare joins between exposures, counterparties, emissions, investments, and related tables.  
This step validates the relationships needed for financed-emissions attribution and reporting evidence assembly.


In [11]:
print(tables["counterparty_emissions"].columns.tolist())
print(tables["counterparty_emissions"].head(2).to_string())

['emissions_id', 'counterparty_id', 'reporting_year', 'scope_1_tco2e', 'scope_2_location_tco2e', 'scope_2_market_tco2e', 'scope_3_tco2e', 'total_ghg_tco2e', 'pcaf_data_quality_score', 'data_source', 'verification_status', 'gwp_basis']
  emissions_id counterparty_id  reporting_year  scope_1_tco2e  scope_2_location_tco2e  scope_2_market_tco2e  scope_3_tco2e  total_ghg_tco2e  pcaf_data_quality_score         data_source verification_status gwp_basis
0     EM000001         CP00001            2022         5407.3                  1966.3                1413.5        14424.7          21798.3                        4  estimated_economic             modeled  IPCC_AR6
1     EM000002         CP00001            2023         5263.0                  1913.8                1883.6        15800.0          22976.8                        4  estimated_economic             modeled  IPCC_AR6


In [12]:
print(tables["counterparties"].columns.tolist())

['counterparty_id', 'bank_id', 'legal_name', 'nace_code', 'nace_description', 'sector_carbon_tier', 'country', 'country_name', 'annual_revenue_meur', 'evic_meur', 'is_listed', 'sme_flag', 'onboarded_date', 'ppp_adjusted_gdp_meur', 'national_scope1_tco2e', 'national_scope2_tco2e', 'national_scope3_tco2e', 'sbti_target_flag', 'data_source_type', 'transition_risk_score']


In [13]:
exp = tables["exposures"].copy()
cp_em = tables["counterparty_emissions"].copy()
cp = tables["counterparties"].copy()

# Enrich counterparty_emissions with EVIC from counterparties
cp_em_enriched = cp_em.merge(
    cp[["counterparty_id", "evic_meur", "ppp_adjusted_gdp_meur", 
        "national_scope1_tco2e", "nace_code"]],
    on="counterparty_id",
    how="left"
)

# Filter to 2024 only
cp_em_2024 = cp_em_enriched[cp_em_enriched["reporting_year"] == 2024].copy()

# Join exposures to enriched emissions
exp_em = exp.merge(
    cp_em_2024[[
        "counterparty_id",
        "total_ghg_tco2e",
        "evic_meur",
        "ppp_adjusted_gdp_meur",
        "national_scope1_tco2e",
        "nace_code"
    ]],
    on="counterparty_id",
    how="left"
)

# Separate corporate vs sovereign exposures
# Sovereigns are NACE O84
corporate_mask = exp_em["nace_code"] != "O84"
sovereign_mask = exp_em["nace_code"] == "O84"

# --- Corporate PCAF B62: outstanding / EVIC * total_ghg ---
exp_em.loc[corporate_mask, "attribution_factor"] = (
    exp_em.loc[corporate_mask, "outstanding_amount_meur"] /
    exp_em.loc[corporate_mask, "evic_meur"]
)
exp_em.loc[corporate_mask, "attributed_emissions_tco2e"] = (
    exp_em.loc[corporate_mask, "attribution_factor"] *
    exp_em.loc[corporate_mask, "total_ghg_tco2e"]
)

# --- Sovereign PCAF B62A: outstanding / ppp_gdp * national_scope1 ---
exp_em.loc[sovereign_mask, "attribution_factor"] = (
    exp_em.loc[sovereign_mask, "outstanding_amount_meur"] /
    exp_em.loc[sovereign_mask, "ppp_adjusted_gdp_meur"]
)
exp_em.loc[sovereign_mask, "attributed_emissions_tco2e"] = (
    exp_em.loc[sovereign_mask, "attribution_factor"] *
    exp_em.loc[sovereign_mask, "national_scope1_tco2e"]
)

print("=== CORPORATE LOANS PCAF ===")
print(f"Total exposure rows: {len(exp_em)}")
print(f"Corporate rows: {corporate_mask.sum()}")
print(f"Sovereign rows: {sovereign_mask.sum()}")
print(f"Matched (non-null attributed): {exp_em['attributed_emissions_tco2e'].notna().sum()}")
print(f"Unmatched: {exp_em['attributed_emissions_tco2e'].isna().sum()}")
print("\nAttributed emissions by bank (tCO2e):")
print(exp_em.groupby("bank_id")["attributed_emissions_tco2e"].sum().round(2))

=== CORPORATE LOANS PCAF ===
Total exposure rows: 603
Corporate rows: 603
Sovereign rows: 0
Matched (non-null attributed): 603
Unmatched: 0

Attributed emissions by bank (tCO2e):
bank_id
BANK01    35973167.69
BANK02    10490960.39
BANK03     6385543.30
BANK04     1078857.69
BANK05     6506404.39
Name: attributed_emissions_tco2e, dtype: float64


In [14]:
inv = tables["investments"].copy()
inv = inv[inv["reporting_year"] == 2024].copy()

# Enrich investments with sovereign data from counterparties
# (for sovereign bonds where ppp_gdp and national emissions are needed)
# For corporate investments, issuer_evic_meur and issuer_revenue_meur are already in investments

listed_equity = inv[inv["asset_class"] == "listed_equity"].copy()
sovereign_bonds = inv[inv["asset_class"] == "sovereign_bond"].copy()

# Listed equity: market_value / issuer_evic * issuer_revenue (PCAF B61)
listed_equity["attribution_factor"] = (
    listed_equity["market_value_meur"] /
    listed_equity["issuer_evic_meur"]
)
listed_equity["attributed_emissions_tco2e"] = (
    listed_equity["attribution_factor"] *
    listed_equity["issuer_revenue_meur"]
).fillna(0)

# Sovereign bonds: nominal / ppp_gdp * national_scope1 (PCAF B62A)
# ppp_gdp_meur is already in investments table
sovereign_bonds["attribution_factor"] = (
    sovereign_bonds["nominal_amount_meur"] /
    sovereign_bonds["ppp_gdp_meur"]
)
# For national emissions, join from counterparties using country code
# sovereigns in counterparties have NACE O84
sovereign_cp = cp[cp["nace_code"] == "O84"][
    ["country", "national_scope1_tco2e", "ppp_adjusted_gdp_meur"]
].copy()

sovereign_bonds = sovereign_bonds.merge(
    sovereign_cp,
    on="country",
    how="left"
)
sovereign_bonds["attributed_emissions_tco2e"] = (
    sovereign_bonds["attribution_factor"] *
    sovereign_bonds["national_scope1_tco2e"]
).fillna(0)

print("=== INVESTMENTS PCAF ===")
print(f"Listed equity rows: {len(listed_equity)}")
print(f"Sovereign bond rows: {len(sovereign_bonds)}")
print("\nListed equity attributed by bank:")
print(listed_equity.groupby("bank_id")["attributed_emissions_tco2e"].sum().round(2))
print("\nSovereign bonds attributed by bank:")
print(sovereign_bonds.groupby("bank_id")["attributed_emissions_tco2e"].sum().round(2))


=== INVESTMENTS PCAF ===
Listed equity rows: 39
Sovereign bond rows: 44

Listed equity attributed by bank:
bank_id
BANK01    3016.52
BANK02    4023.74
BANK03    1172.65
BANK04    1765.92
BANK05     304.04
Name: attributed_emissions_tco2e, dtype: float64

Sovereign bonds attributed by bank:
bank_id
BANK01    1044125.10
BANK02      70413.48
BANK03     721512.55
BANK04     390119.74
BANK05     372598.37
Name: attributed_emissions_tco2e, dtype: float64


## 2.6 KPI and Metric Calculation

Aggregate emissions and financial information at bank and reporting-year level.  
This includes financed emissions, carbon intensity, investment emissions, green-finance indicators, and climate-related reporting metrics.


In [15]:
fs = tables["financial_summary"].copy()

# Aggregate all sources
corp_totals = exp_em.groupby("bank_id")["attributed_emissions_tco2e"].sum().reset_index()
corp_totals.columns = ["bank_id", "financed_em_loans_tco2e"]

eq_totals = listed_equity.groupby("bank_id")["attributed_emissions_tco2e"].sum().reset_index()
eq_totals.columns = ["bank_id", "financed_em_equity_tco2e"]

sov_totals = sovereign_bonds.groupby("bank_id")["attributed_emissions_tco2e"].sum().reset_index()
sov_totals.columns = ["bank_id", "financed_em_sovereign_tco2e"]

# Build master financed emissions table
fin_em = (
    tables["banks"][["bank_id", "total_loans_meur"]]
    .merge(corp_totals, on="bank_id", how="left")
    .merge(eq_totals, on="bank_id", how="left")
    .merge(sov_totals, on="bank_id", how="left")
    .fillna(0)
)

fin_em["total_financed_emissions_tco2e"] = (
    fin_em["financed_em_loans_tco2e"] +
    fin_em["financed_em_equity_tco2e"] +
    fin_em["financed_em_sovereign_tco2e"]
)

# Carbon intensity — the blank field from financial_summary
fin_em["carbon_intensity_tco2e_per_meur_lending"] = (
    fin_em["total_financed_emissions_tco2e"] /
    fin_em["total_loans_meur"]
).round(4)

print("=== FINAL FINANCED EMISSIONS ===")
print(fin_em[[
    "bank_id",
    "financed_em_loans_tco2e",
    "financed_em_equity_tco2e",
    "financed_em_sovereign_tco2e",
    "total_financed_emissions_tco2e",
    "carbon_intensity_tco2e_per_meur_lending"
]].round(2).to_string())

# Push carbon intensity back into financial_summary
fs = fs.merge(
    fin_em[["bank_id", 
            "total_financed_emissions_tco2e",
            "carbon_intensity_tco2e_per_meur_lending"]],
    on="bank_id",
    how="left",
    suffixes=("_old", "")
)

# Drop the old blank column if it exists
if "carbon_intensity_tco2e_per_meur_lending_old" in fs.columns:
    fs.drop(columns=["carbon_intensity_tco2e_per_meur_lending_old"], inplace=True)

print("\n=== FINANCIAL SUMMARY — carbon intensity filled ===")
print(fs[["bank_id", "reporting_year", 
          "carbon_intensity_tco2e_per_meur_lending"]].to_string())

# Save
fs.to_csv("financial_summary_clean.csv", index=False)
print("\nfinancial_summary_clean.csv saved")

=== FINAL FINANCED EMISSIONS ===
  bank_id  financed_em_loans_tco2e  financed_em_equity_tco2e  financed_em_sovereign_tco2e  total_financed_emissions_tco2e  carbon_intensity_tco2e_per_meur_lending
0  BANK01              35973167.69                   3016.52                   1044125.10                     37020309.30                                  1188.43
1  BANK02              10490960.39                   4023.74                     70413.48                     10565397.62                                   537.99
2  BANK03               6385543.30                   1172.65                    721512.55                      7108228.50                                   295.09
3  BANK04               1078857.69                   1765.92                    390119.74                      1470743.35                                   165.43
4  BANK05               6506404.39                    304.04                    372598.37                      6879306.80                               

In [16]:
# Recompute per bank per year
results = []

for year in [2022, 2023, 2024]:
    cp_em_year = cp_em_enriched[cp_em_enriched["reporting_year"] == year].copy()
    
    exp_em_year = exp.merge(
        cp_em_year[[
            "counterparty_id", "total_ghg_tco2e",
            "evic_meur", "ppp_adjusted_gdp_meur",
            "national_scope1_tco2e", "nace_code"
        ]],
        on="counterparty_id",
        how="left"
    )
    
    corporate_mask = exp_em_year["nace_code"] != "O84"
    sovereign_mask = exp_em_year["nace_code"] == "O84"

    exp_em_year.loc[corporate_mask, "attributed_emissions_tco2e"] = (
        exp_em_year.loc[corporate_mask, "outstanding_amount_meur"] /
        exp_em_year.loc[corporate_mask, "evic_meur"] *
        exp_em_year.loc[corporate_mask, "total_ghg_tco2e"]
    )
    exp_em_year.loc[sovereign_mask, "attributed_emissions_tco2e"] = (
        exp_em_year.loc[sovereign_mask, "outstanding_amount_meur"] /
        exp_em_year.loc[sovereign_mask, "ppp_adjusted_gdp_meur"] *
        exp_em_year.loc[sovereign_mask, "national_scope1_tco2e"]
    )
    
    year_totals = exp_em_year.groupby("bank_id")["attributed_emissions_tco2e"].sum().reset_index()
    year_totals["reporting_year"] = year
    results.append(year_totals)

financed_by_year = pd.concat(results, ignore_index=True)
financed_by_year.columns = ["bank_id", "financed_em_loans_tco2e", "reporting_year"]

# Merge with banks to get total_loans denominator
financed_by_year = financed_by_year.merge(
    tables["banks"][["bank_id", "total_loans_meur"]],
    on="bank_id", how="left"
)

financed_by_year["carbon_intensity_tco2e_per_meur_lending"] = (
    financed_by_year["financed_em_loans_tco2e"] /
    financed_by_year["total_loans_meur"]
).round(4)

print("=== CARBON INTENSITY BY YEAR ===")
print(financed_by_year[[
    "bank_id", "reporting_year", 
    "financed_em_loans_tco2e",
    "carbon_intensity_tco2e_per_meur_lending"
]].to_string())

# Push into financial_summary properly
fs_clean = tables["financial_summary"].copy()

# Drop old blank column
if "carbon_intensity_tco2e_per_meur_lending" in fs_clean.columns:
    fs_clean.drop(columns=["carbon_intensity_tco2e_per_meur_lending"], inplace=True)

fs_clean = fs_clean.merge(
    financed_by_year[["bank_id", "reporting_year", 
                       "carbon_intensity_tco2e_per_meur_lending",
                       "financed_em_loans_tco2e"]],
    on=["bank_id", "reporting_year"],
    how="left"
)

print("\n=== FINAL FINANCIAL SUMMARY ===")
print(fs_clean[["bank_id", "reporting_year", 
                "carbon_intensity_tco2e_per_meur_lending"]].to_string())

fs_clean.to_csv("financial_summary_clean.csv", index=False)
print("\nfinancial_summary_clean.csv saved")

=== CARBON INTENSITY BY YEAR ===
   bank_id  reporting_year  financed_em_loans_tco2e  carbon_intensity_tco2e_per_meur_lending
0   BANK01            2022             3.655018e+07                                1173.3363
1   BANK02            2022             1.217838e+07                                 620.1217
2   BANK03            2022             7.110264e+06                                 295.1703
3   BANK04            2022             1.273748e+06                                 143.2694
4   BANK05            2022             6.444307e+06                                 658.6981
5   BANK01            2023             3.742759e+07                                1201.5032
6   BANK02            2023             1.125597e+07                                 573.1527
7   BANK03            2023             6.284511e+06                                 260.8907
8   BANK04            2023             1.200798e+06                                 135.0641
9   BANK05            2023           

## 2.7 Payload Preparation

Generate full bank-level payloads and section-specific payloads for the reporting agents.  
The active exporter writes strict JSON, converts missing values to `null`, includes Governance and Strategy evidence, and validates every exported file.


### Deprecated exporter cell disabled

This older export pipeline was disabled to avoid producing non-strict JSON or outdated payload files. 
Use the final patched exporter cell below instead.


### Final patched exporter

This cell is the active export pipeline. It writes strict JSON with missing values as `null`, 
includes `climate_risk_register` in Governance payloads, includes `targets` in Strategy payloads, 
and validates all exported payload files after writing.


In [17]:
# 2.7 Final ESG payload preparation pipeline — IFRS S1/S2 Banking Dataset
# ============================================================

import pandas as pd
import json
import os
from pathlib import Path

# ── 0. LOAD TABLES ───────────────────────────────────────────
BASE_DIR = Path.cwd()
if not (BASE_DIR / "gen_data").exists() and (BASE_DIR / "notebooks" / "gen_data").exists():
    BASE_DIR = BASE_DIR / "notebooks"
DATA_PATH = BASE_DIR / "gen_data"
OUTPUT_PATH = BASE_DIR.parent / "payloads" if BASE_DIR.name == "notebooks" else BASE_DIR / "payloads"

tables = {}
for f in os.listdir(DATA_PATH):
    if f.endswith(".csv"):
        tables[f.replace(".csv", "")] = pd.read_csv(DATA_PATH / f)

# Override vehicles with validated clean version
tables["vehicles"] = pd.read_csv(DATA_PATH / "vehicles.csv")

# ← NEW: ensure enriched tables from fix cells are loaded
# If cells 7b/7c/7d ran before this cell, gen_data/ already has the new files.
# These three lines guarantee we always load the latest versions.
tables["disclosures"] = pd.read_csv(DATA_PATH / "disclosures.csv")
tables["climate_opportunities"] = pd.read_csv(DATA_PATH / "climate_opportunities.csv")
tables["climate_scenarios"] = pd.read_csv(DATA_PATH / "climate_scenarios.csv")

print(f"Loaded {len(tables)} tables")
print(f"  disclosures: {len(tables['disclosures'])} rows")
print(f"  climate_opportunities: {len(tables['climate_opportunities'])} rows")
print(f"  climate_scenarios: {len(tables['climate_scenarios'])} rows (has methodology_notes: "
      f"{'methodology_notes' in tables['climate_scenarios'].columns})")


# ── 1. PARSE JSON FIELDS ─────────────────────────────────────
targets = tables["targets"].copy()
targets["milestones_parsed"] = targets["interim_milestones_json"].apply(
    lambda x: json.loads(x) if pd.notna(x) else []
)

# Honest progress descriptor.
# IMPORTANT: schedule-elapsed (time fraction) is NOT emission-reduction achievement.
# We expose both, and a basis flag, so downstream prose never presents the time
# fraction as achieved reduction. No fabrication: actual progress stays None unless
# a current actual value on the SAME scale as the baseline is genuinely available.
def compute_target_progress(row):
    try:
        baseline_yr = int(row.get("baseline_year", 2019))
        target_yr   = int(row.get("target_year", 2030))
    except Exception:
        baseline_yr, target_yr = None, None

    schedule = None
    if baseline_yr is not None and target_yr is not None and target_yr > baseline_yr:
        schedule = round(min((2024 - baseline_yr) / (target_yr - baseline_yr) * 100, 100.0), 1)

    # Real progress requires a validated current actual on the baseline's scale.
    # None is available for these target metrics on a matching scale, so we do NOT
    # manufacture an achievement figure.
    actual = None

    if actual is not None:
        basis = "actual_vs_baseline"
    elif schedule is not None:
        basis = "schedule_elapsed_only"
    else:
        basis = "unavailable"

    return {
        "schedule_elapsed_pct_2024":   schedule,
        "actual_progress_pct_2024":    actual,
        "progress_basis":              basis,
        "progress_is_schedule_proxy":  (actual is None and schedule is not None),
    }

_prog = targets.apply(compute_target_progress, axis=1, result_type="expand")
for _c in ["schedule_elapsed_pct_2024","actual_progress_pct_2024","progress_basis","progress_is_schedule_proxy"]:
    targets[_c] = _prog[_c]
# Backwards-compatible field, but now ONLY reflects real achievement (None when we only
# know schedule elapsed). This prevents the time fraction leaking out as "progress".
targets["target_progress_pct_2024"] = targets["actual_progress_pct_2024"]
print("Targets: milestones parsed. Progress computed honestly (schedule vs actual separated).")
print(targets[["target_id","bank_id","target_year","schedule_elapsed_pct_2024",
               "actual_progress_pct_2024","progress_basis"]].to_string())


# ── 2. STANDARDISE DATES ─────────────────────────────────────
travel = tables["travel_records"].copy()
travel["travel_date"] = pd.to_datetime(travel["travel_date"])
travel["reporting_year"] = travel["travel_date"].dt.year

util = tables["utility_invoices"].copy()
util["invoice_month"] = pd.to_datetime(util["invoice_month"])

print("Dates standardised")


# ── 3. SCOPE AGGREGATIONS ────────────────────────────────────

# Scope 1 — gas from utility invoices
scope1_gas = util.groupby(["bank_id", "invoice_year"])["scope1_gas_tco2e"].sum().reset_index()
scope1_gas.columns = ["bank_id", "reporting_year", "scope1_gas_tco2e"]

# Scope 1 — fleet from vehicles (2024 only, uses corrected field)
scope1_fleet = (
    tables["vehicles"]
    .groupby("bank_id")["scope1_tco2e_2024_clean"]
    .sum()
    .reset_index()
)
scope1_fleet.columns = ["bank_id", "scope1_fleet_tco2e"]
scope1_fleet["reporting_year"] = 2024

scope1 = scope1_gas.merge(
    scope1_fleet, on=["bank_id", "reporting_year"], how="left"
).fillna(0)
scope1["scope1_total_tco2e"] = scope1["scope1_gas_tco2e"] + scope1["scope1_fleet_tco2e"]
scope1["fleet_data_available"] = scope1["reporting_year"] == 2024

# Scope 2 — location and market based
# ← NOTE on REC reconciliation (documented here to prevent audit confusion):
# scope2_market_tco2e in utility_invoices already reflects market-based accounting
# with grid emission factors adjusted for contractual instruments (RECs/GOs).
# The rec_registry.csv is NOT subtracted again here — it is used only for
# narrative disclosure of certificate type and volume. There is NO double-counting.
scope2 = util.groupby(["bank_id", "invoice_year"]).agg(
    scope2_location_tco2e=("scope2_location_tco2e", "sum"),
    scope2_market_tco2e=("scope2_market_tco2e", "sum")
).reset_index().rename(columns={"invoice_year": "reporting_year"})

# ← NEW: add a reconciliation check — market should be <= location (RECs reduce market Scope 2)
scope2_check = scope2[scope2["reporting_year"] == 2024].copy()
assert (scope2_check["scope2_market_tco2e"] <= scope2_check["scope2_location_tco2e"]).all(), \
    "WARNING: scope2_market > scope2_location for some banks — verify REC treatment in invoices"
print("Scope 2 REC reconciliation check passed (market <= location for all banks in 2024)")

# Scope 3 Cat.6 — business travel
scope3_travel = (
    travel.groupby(["bank_id", "reporting_year"])["emissions_kg_co2e"]
    .sum()
    .reset_index()
)
scope3_travel.columns = ["bank_id", "reporting_year", "scope3_travel_tco2e"]
scope3_travel["scope3_travel_tco2e"] = (scope3_travel["scope3_travel_tco2e"] / 1000).round(4)
scope3_travel["travel_data_available"] = scope3_travel["reporting_year"] == 2024

print("Scope 1 / 2 / 3 aggregations done")


# ── 4. PCAF FINANCED EMISSIONS — CORPORATE LOANS ─────────────
exp = tables["exposures"].copy()
cp = tables["counterparties"].copy()
cp_em = tables["counterparty_emissions"].copy()

cp_em_enriched = cp_em.merge(
    cp[["counterparty_id", "evic_meur", "ppp_adjusted_gdp_meur",
        "national_scope1_tco2e", "nace_code"]],
    on="counterparty_id",
    how="left"
)

results = []
for year in [2022, 2023, 2024]:
    cp_em_year = cp_em_enriched[cp_em_enriched["reporting_year"] == year].copy()

    exp_em_year = exp.merge(
        cp_em_year[[
            "counterparty_id", "total_ghg_tco2e",
            "evic_meur", "ppp_adjusted_gdp_meur",
            "national_scope1_tco2e", "nace_code"
        ]],
        on="counterparty_id",
        how="left"
    )

    corporate_mask = exp_em_year["nace_code"] != "O84"
    sovereign_mask = exp_em_year["nace_code"] == "O84"

    exp_em_year.loc[corporate_mask, "attributed_emissions_tco2e"] = (
        exp_em_year.loc[corporate_mask, "outstanding_amount_meur"] /
        exp_em_year.loc[corporate_mask, "evic_meur"] *
        exp_em_year.loc[corporate_mask, "total_ghg_tco2e"]
    )
    exp_em_year.loc[sovereign_mask, "attributed_emissions_tco2e"] = (
        exp_em_year.loc[sovereign_mask, "outstanding_amount_meur"] /
        exp_em_year.loc[sovereign_mask, "ppp_adjusted_gdp_meur"] *
        exp_em_year.loc[sovereign_mask, "national_scope1_tco2e"]
    )

    year_totals = exp_em_year.groupby("bank_id")["attributed_emissions_tco2e"].sum().reset_index()
    year_totals["reporting_year"] = year
    results.append(year_totals)

financed_by_year = pd.concat(results, ignore_index=True)
financed_by_year.columns = ["bank_id", "financed_em_loans_tco2e", "reporting_year"]
financed_by_year = financed_by_year.merge(
    tables["banks"][["bank_id", "total_loans_meur"]], on="bank_id", how="left"
)
financed_by_year["carbon_intensity_tco2e_per_meur_lending"] = (
    financed_by_year["financed_em_loans_tco2e"] /
    financed_by_year["total_loans_meur"]
).round(4)

print(f"PCAF corporate loans: computed for {len(financed_by_year)} bank-year rows")


# ── 5. PCAF INVESTMENTS ──────────────────────────────────────
inv = tables["investments"].copy()
inv = inv[inv["reporting_year"] == 2024].copy()

listed_equity = inv[inv["asset_class"] == "listed_equity"].copy()
sovereign_bonds = inv[inv["asset_class"] == "sovereign_bond"].copy()

listed_equity["attribution_factor"] = (
    listed_equity["market_value_meur"] /
    listed_equity["issuer_evic_meur"]
)
listed_equity["attributed_emissions_proxy_tco2e"] = (
    listed_equity["attribution_factor"] *
    listed_equity["issuer_revenue_meur"]
).fillna(0)
listed_equity["emissions_proxy_used"] = True
listed_equity["proxy_basis"] = "issuer_revenue_meur"
listed_equity["proxy_confidence"] = "low"
listed_equity["proxy_reason"] = (
    "Direct issuer emissions unavailable. Revenue used as PCAF B61 proxy. "
    "Do not treat as verified emissions."
)

sovereign_bonds["attribution_factor"] = (
    sovereign_bonds["nominal_amount_meur"] /
    sovereign_bonds["ppp_gdp_meur"]
)
sovereign_cp = cp[cp["nace_code"] == "O84"][
    ["country", "national_scope1_tco2e", "ppp_adjusted_gdp_meur"]
].copy()
sovereign_bonds = sovereign_bonds.merge(sovereign_cp, on="country", how="left")
sovereign_bonds["attributed_emissions_tco2e"] = (
    sovereign_bonds["attribution_factor"] *
    sovereign_bonds["national_scope1_tco2e"]
)
sovereign_bonds["data_gap_flag"] = sovereign_bonds["national_scope1_tco2e"].isna()
sovereign_bonds["data_gap_reason"] = sovereign_bonds["data_gap_flag"].apply(
    lambda x: "Missing sovereign national emissions — attribution not calculable" if x else None
)

print("PCAF investments done")


# ── 6. COMPLETE FINANCIAL SUMMARY ────────────────────────────
fs = tables["financial_summary"].copy()

if "carbon_intensity_tco2e_per_meur_lending" in fs.columns:
    fs.drop(columns=["carbon_intensity_tco2e_per_meur_lending"], inplace=True)

fs = fs.merge(
    financed_by_year[[
        "bank_id", "reporting_year",
        "carbon_intensity_tco2e_per_meur_lending",
        "financed_em_loans_tco2e"
    ]],
    on=["bank_id", "reporting_year"],
    how="left"
)

print("financial_summary: carbon intensity populated")


# ── NEW: Precompute sector exposure KPIs ──────────────────────────────────
# ← NEW FIX: high_carbon_sector_exposure_pct, fossil_fuel_exposure_pct
# These are needed for the Strategy and Risk Management sections.
# High-carbon NACE codes: B (mining), C19 (petroleum), C20 (chemicals),
#   D35 (electricity/gas), H49 (land transport), H50 (water transport), H51 (air transport)
# Fossil fuel NACE: B06 (crude oil), B07 (metal ores/coal proxy), C19, D35

HIGH_CARBON_NACE = {"B06","B07","B08","B09","C19","C20","D35","H49","H50","H51","C24"}
FOSSIL_FUEL_NACE = {"B06","B07","C19","D35"}

exp_cp = exp.merge(cp[["counterparty_id","nace_code"]], on="counterparty_id", how="left")

sector_kpis = {}
for bank_id in tables["banks"]["bank_id"].unique():
    bank_exp = exp_cp[exp_cp["bank_id"] == bank_id]
    total_outstanding = bank_exp["outstanding_amount_meur"].sum()
    if total_outstanding == 0:
        sector_kpis[bank_id] = {"high_carbon_sector_exposure_pct": 0.0,
                                  "fossil_fuel_exposure_pct": 0.0}
        continue

    hc_exp = bank_exp[bank_exp["nace_code"].isin(HIGH_CARBON_NACE)]["outstanding_amount_meur"].sum()
    ff_exp = bank_exp[bank_exp["nace_code"].isin(FOSSIL_FUEL_NACE)]["outstanding_amount_meur"].sum()

    sector_kpis[bank_id] = {
        "high_carbon_sector_exposure_pct": round(hc_exp / total_outstanding * 100, 2),
        "fossil_fuel_exposure_pct":        round(ff_exp / total_outstanding * 100, 2),
        "high_carbon_sector_exposure_meur": round(hc_exp, 2),
        "fossil_fuel_exposure_meur":        round(ff_exp, 2),
    }

print("\n=== SECTOR EXPOSURE KPIs ===")
for b, v in sector_kpis.items():
    print(f"  {b}: high_carbon={v['high_carbon_sector_exposure_pct']}%  "
          f"fossil_fuel={v['fossil_fuel_exposure_pct']}%")


# ← NEW FIX: emissions data quality summary per bank
# Aggregates counterparty data_source_type distribution weighted by outstanding exposure
def build_data_quality_summary(bank_id):
    bank_exp = exp_cp[exp_cp["bank_id"] == bank_id].copy()
    bank_exp = bank_exp.merge(
        cp[["counterparty_id","data_source_type"]],
        on="counterparty_id", how="left"
    )
    total = bank_exp["outstanding_amount_meur"].sum()
    if total == 0:
        return {}
    summary = (
        bank_exp.groupby("data_source_type")["outstanding_amount_meur"]
        .sum()
        .div(total)
        .mul(100)
        .round(1)
        .to_dict()
    )
    return summary

dq_summaries = {b: build_data_quality_summary(b) for b in tables["banks"]["bank_id"].unique()}

print("\n=== EMISSIONS DATA QUALITY SUMMARY (% of exposure by data source type) ===")
for b, v in dq_summaries.items():
    print(f"  {b}: {v}")


# ── 6B. SOURCE COHERENCE FIXES BEFORE PAYLOAD EXPORT ──────────
# These fixes are dynamic and derived from calculated data. They are not manual
# output patches: they correct in-memory source tables before the payloads are built.

coherence_fixes_applied = []

def _safe_float(v):
    try:
        f = float(v)
        return f if f == f else None
    except Exception:
        return None

def _record_fix(bank_id, table, field, old_value, new_value, reason):
    if old_value != new_value:
        coherence_fixes_applied.append({
            "bank_id": bank_id,
            "table": table,
            "field": field,
            "old_value": old_value,
            "new_value": new_value,
            "reason": reason,
        })

# 6B.1 Align bank-level financial fields with the 2024 financial_summary row.
for _idx, _bank in tables["banks"].iterrows():
    _bid = _bank["bank_id"]
    _fs24 = fs[(fs["bank_id"] == _bid) & (fs["reporting_year"] == 2024)]
    if _fs24.empty:
        continue
    _fs24 = _fs24.iloc[0]
    for _col in ["total_assets_meur", "total_loans_meur", "tier1_capital_meur", "cet1_ratio_pct"]:
        if _col in tables["banks"].columns and _col in fs.columns:
            _old = tables["banks"].at[_idx, _col]
            _new = _fs24[_col]
            if pd.notna(_new) and _old != _new:
                tables["banks"].at[_idx, _col] = _new
                _record_fix(_bid, "banks", _col, _old, _new, "Aligned bank master field to 2024 financial_summary.")

# 6B.2 Fix financed-emissions intensity targets using the computed baseline-year intensity.
# If the old target baseline was on the wrong scale, milestones are rescaled by the same ratio.
for _idx, _t in targets.iterrows():
    _bid = _t.get("bank_id")
    _metric = str(_t.get("metric", "")).lower()
    _ttype = str(_t.get("target_type", "")).lower()
    _scope = str(_t.get("scope", "") or _t.get("scope_coverage", "")).lower()
    _is_financed_intensity = (
        ("intensity" in _ttype or "per_meur" in _metric or "per_m" in _metric)
        and ("financed" in _scope or "cat15" in _scope or "scope_3" in _scope or "scope3" in _scope)
    )
    if not _is_financed_intensity:
        continue

    _baseline_year = int(_t.get("baseline_year", 2022)) if pd.notna(_t.get("baseline_year", None)) else 2022
    _calc = financed_by_year[(financed_by_year["bank_id"] == _bid) & (financed_by_year["reporting_year"] == _baseline_year)]
    if _calc.empty:
        _calc = financed_by_year[financed_by_year["bank_id"] == _bid].sort_values("reporting_year")
        if _calc.empty:
            continue
        _baseline_year = int(_calc.iloc[0]["reporting_year"])
        targets.at[_idx, "baseline_year"] = _baseline_year

    _computed_baseline = float(_calc.iloc[0]["carbon_intensity_tco2e_per_meur_lending"])
    _old_baseline = _safe_float(_t.get("baseline_value"))

    if _old_baseline and (_computed_baseline / _old_baseline > 10 or _computed_baseline / _old_baseline < 0.1):
        _ratio = _computed_baseline / _old_baseline
        targets.at[_idx, "baseline_value"] = round(_computed_baseline, 4)
        targets.at[_idx, "baseline_adjusted_flag"] = True
        targets.at[_idx, "baseline_adjustment_note"] = (
            "Baseline rescaled to computed financed-emissions intensity for the baseline year. "
            "The previous value was not on the same tCO2e/EURm lending scale."
        )
        _record_fix(_bid, "targets", "baseline_value", _old_baseline, round(_computed_baseline, 4),
                    "Rescaled financed-emissions intensity baseline to computed value.")

        _milestones = _t.get("milestones_parsed", [])
        if isinstance(_milestones, list):
            _new_milestones = []
            for _m in _milestones:
                _m2 = dict(_m)
                if _safe_float(_m2.get("value")) is not None:
                    _m2["value"] = round(float(_m2["value"]) * _ratio, 4)
                _new_milestones.append(_m2)
            targets.at[_idx, "milestones_parsed"] = _new_milestones
            targets.at[_idx, "interim_milestones_json"] = json.dumps(_new_milestones)

        _reduction = _safe_float(_t.get("target_value_pct_reduction"))
        if _reduction is not None and "target_value" not in targets.columns:
            targets["target_value"] = None
        if _reduction is not None:
            _target_value = round(_computed_baseline * (1 - _reduction / 100), 4)
            targets.at[_idx, "target_value"] = _target_value
            if "target_intensity_tco2e_per_meur" in tables["banks"].columns:
                _bmask = tables["banks"]["bank_id"] == _bid
                _old_bank_target = tables["banks"].loc[_bmask, "target_intensity_tco2e_per_meur"].iloc[0]
                tables["banks"].loc[_bmask, "target_intensity_tco2e_per_meur"] = _target_value
                _record_fix(_bid, "banks", "target_intensity_tco2e_per_meur", _old_bank_target, _target_value,
                            "Aligned bank target intensity to corrected financed-emissions target value.")

# 6B.3 Fix all-scopes / net-zero target baselines that are below financed emissions alone.
for _idx, _t in targets.iterrows():
    _bid = _t.get("bank_id")
    _scope = str(_t.get("scope", "") or _t.get("scope_coverage", "")).lower()
    _ttype = str(_t.get("target_type", "")).lower()
    _metric = str(_t.get("metric", "")).lower()
    _is_intensity = "intensity" in _ttype or "per_meur" in _metric
    _is_broad = ("all" in _scope and "scope" in _scope) or ("net" in _ttype and "zero" in _ttype)
    _old_base = _safe_float(_t.get("baseline_value"))
    if not _is_broad or _is_intensity or _old_base is None:
        continue
    _bank_fin = financed_by_year[financed_by_year["bank_id"] == _bid].sort_values("reporting_year")
    if _bank_fin.empty:
        continue
    _base_year = int(_bank_fin.iloc[0]["reporting_year"])
    _fin_base = float(_bank_fin.iloc[0]["financed_em_loans_tco2e"])
    _s1_base = scope1[(scope1["bank_id"] == _bid) & (scope1["reporting_year"] == _base_year)]
    _s2_base = scope2[(scope2["bank_id"] == _bid) & (scope2["reporting_year"] == _base_year)]
    _op_base = 0.0
    if not _s1_base.empty:
        _op_base += float(_s1_base.iloc[0]["scope1_total_tco2e"])
    if not _s2_base.empty:
        _op_base += float(_s2_base.iloc[0]["scope2_market_tco2e"])
    _new_base = round(_fin_base + _op_base, 4)
    if _old_base < _fin_base:
        targets.at[_idx, "baseline_year"] = _base_year
        targets.at[_idx, "baseline_value"] = _new_base
        targets.at[_idx, "baseline_adjusted_flag"] = True
        targets.at[_idx, "baseline_adjustment_note"] = (
            "Baseline reset to earliest available financed-emissions baseline plus operational emissions. "
            "Previous broad-scope baseline was smaller than financed emissions alone."
        )
        _record_fix(_bid, "targets", "baseline_value", _old_base, _new_base,
                    "Corrected broad-scope baseline to include financed emissions.")

# 6B.4 Make board percentage fields compatible with whole director counts.
if "governance" in tables and "board_size" in tables["governance"].columns:
    gov = tables["governance"].copy()
    for _idx, _g in gov.iterrows():
        _bid = _g.get("bank_id")
        _bs = _safe_float(_g.get("board_size"))
        if not _bs:
            continue
        for _col in ["independent_directors_pct", "board_climate_expertise_pct"]:
            if _col not in gov.columns:
                continue
            _old = _safe_float(_g.get(_col))
            if _old is None:
                continue
            _director_count = round(_old / 100 * _bs)
            _new = round(_director_count / _bs * 100, 1)
            if abs(_old - _new) > 0.01:
                gov.at[_idx, _col] = _new
                _record_fix(_bid, "governance", _col, _old, _new,
                            "Rounded percentage to a whole-number director count.")

    # Recompute climate agenda % from board_minutes_extract where possible.
    bm = tables.get("board_minutes_extract")
    if bm is not None and {"bank_id", "reporting_year", "committee_type", "climate_agenda_flag"}.issubset(bm.columns):
        full_board = bm[bm["committee_type"].astype(str).str.lower().eq("full_board")].copy()
        agenda = full_board.groupby(["bank_id", "reporting_year"])["climate_agenda_flag"].mean().mul(100).round(1).reset_index()
        for _idx, _g in gov.iterrows():
            _match = agenda[(agenda["bank_id"] == _g["bank_id"]) & (agenda["reporting_year"] == _g["reporting_year"])]
            if not _match.empty and "climate_on_board_agenda_pct" in gov.columns:
                _old = _safe_float(_g.get("climate_on_board_agenda_pct"))
                _new = float(_match.iloc[0]["climate_agenda_flag"])
                gov.at[_idx, "climate_on_board_agenda_pct"] = _new
                _record_fix(_g["bank_id"], "governance", "climate_on_board_agenda_pct", _old, _new,
                            "Recomputed from Full Board minutes climate_agenda_flag.")
    tables["governance"] = gov

# 6B.5 Cap/recalculate scenario absolute monetary values to avoid impossible magnitudes.
if "climate_scenarios" in tables:
    cs = tables["climate_scenarios"].copy()
    for _idx, _s in cs.iterrows():
        _bid = _s.get("bank_id")
        _ff_exp = sector_kpis.get(_bid, {}).get("fossil_fuel_exposure_meur")
        _high_risk_pct = _safe_float(_s.get("high_risk_exposure_pct"))
        if _ff_exp is not None and _high_risk_pct is not None and "stranded_assets_estimate_meur" in cs.columns:
            _old = _safe_float(_s.get("stranded_assets_estimate_meur"))
            _new = round(_ff_exp * _high_risk_pct / 100, 2)
            if _old is not None and abs(_old - _new) > 0.01:
                cs.at[_idx, "stranded_assets_estimate_meur"] = _new
                cs.at[_idx, "scenario_magnitude_adjusted_flag"] = True
                _record_fix(_bid, "climate_scenarios", "stranded_assets_estimate_meur", _old, _new,
                            "Recalculated as fossil-fuel exposure multiplied by high-risk exposure percentage.")
        if "revenue_at_risk_meur" in cs.columns:
            _fs24 = fs[(fs["bank_id"] == _bid) & (fs["reporting_year"] == 2024)]
            if not _fs24.empty and "total_revenue_meur" in _fs24.columns:
                _revenue = _safe_float(_fs24.iloc[0].get("total_revenue_meur"))
                _old = _safe_float(_s.get("revenue_at_risk_meur"))
                if _revenue is not None and _old is not None:
                    _cap = round(_revenue * 0.30, 2)
                    _new = min(_old, _cap)
                    if _new != _old:
                        cs.at[_idx, "revenue_at_risk_meur"] = _new
                        cs.at[_idx, "scenario_magnitude_adjusted_flag"] = True
                        _record_fix(_bid, "climate_scenarios", "revenue_at_risk_meur", _old, _new,
                                    "Capped at 30% of 2024 revenue to avoid impossible report magnitudes.")
    tables["climate_scenarios"] = cs

# 6B.6 Improve obvious risk mitigation mismatches without changing risk scores.
def _aligned_mitigation(name, category):
    n = str(name).lower()
    c = str(category).lower()
    if "flood" in n:
        return "Collateral revaluation, flood-zone mapping overlay and updated mortgage underwriting criteria"
    if "wildfire" in n:
        return "Wildfire hazard mapping, collateral revaluation and borrower resilience engagement"
    if "heat" in n or "agric" in n:
        return "Climate scenario stress testing, sector monitoring and borrower adaptation engagement"
    if "stranded" in n or "fossil" in n:
        return "Sector exposure limits, enhanced due diligence and portfolio decarbonisation glide-path monitoring"
    if "carbon pricing" in n or "policy" in c:
        return "Sector exposure limits, carbon-cost sensitivity analysis and enhanced due diligence"
    if "ev shift" in n or "technology" in c:
        return "Technology transition monitoring, sector concentration limits and client transition-plan engagement"
    if "reputational" in c or "scrutiny" in n:
        return "Enhanced financed-emissions disclosure controls and client engagement on transition plans"
    return None

if "climate_risk_register" in tables:
    rr = tables["climate_risk_register"].copy()
    for _idx, _r in rr.iterrows():
        _new = _aligned_mitigation(_r.get("risk_name"), _r.get("risk_category"))
        if _new:
            _old = _r.get("mitigation_actions")
            if _old != _new:
                rr.at[_idx, "mitigation_actions"] = _new
                _record_fix(_r.get("bank_id"), "climate_risk_register", "mitigation_actions", _old, _new,
                            "Aligned mitigation action to risk type.")


    # Recalculate risk_rating from the stated 5x5 risk matrix.
    # This keeps the disclosure data consistent with the methodology:
    # likelihood_score * severity_score; critical >= 15, high >= 8, medium >= 3, low < 3.
    def _risk_rating_from_scores(likelihood, severity):
        try:
            score = float(likelihood) * float(severity)
        except Exception:
            return None
        if score >= 15:
            return "critical"
        if score >= 8:
            return "high"
        if score >= 3:
            return "medium"
        return "low"

    for _idx, _r in rr.iterrows():
        _new_rating = _risk_rating_from_scores(_r.get("likelihood_score"), _r.get("severity_score"))
        if _new_rating is not None:
            _old_rating = _r.get("risk_rating")
            if _old_rating != _new_rating:
                rr.at[_idx, "risk_rating"] = _new_rating
                _record_fix(
                    _r.get("bank_id"),
                    "climate_risk_register",
                    "risk_rating",
                    _old_rating,
                    _new_rating,
                    "Recalculated from likelihood_score * severity_score using the stated 5x5 risk matrix."
                )

    tables["climate_risk_register"] = rr

print("\n=== SOURCE COHERENCE FIXES APPLIED ===")
print(f"Total fixes: {len(coherence_fixes_applied)}")
if coherence_fixes_applied:
    print(pd.DataFrame(coherence_fixes_applied).head(30).to_string(index=False))


# ── 7. KPI BUILDER ────────────────────────────────────────────
def build_reporting_kpis(bank_id):

    def get_2024_value(df, year_col, value_col):
        rows = df[(df["bank_id"] == bank_id) & (df[year_col] == 2024)]
        if rows.empty:
            return None
        val = rows[value_col].values[0]
        return None if pd.isna(val) else round(float(val), 4)

    s1 = scope1[(scope1["bank_id"] == bank_id) & (scope1["reporting_year"] == 2024)]
    scope1_total = float(s1["scope1_total_tco2e"].values[0]) if not s1.empty else None

    s2 = scope2[(scope2["bank_id"] == bank_id) & (scope2["reporting_year"] == 2024)]
    scope2_location = float(s2["scope2_location_tco2e"].values[0]) if not s2.empty else None
    scope2_market   = float(s2["scope2_market_tco2e"].values[0])   if not s2.empty else None

    s3 = scope3_travel[(scope3_travel["bank_id"] == bank_id) & (scope3_travel["reporting_year"] == 2024)]
    scope3 = float(s3["scope3_travel_tco2e"].values[0]) if not s3.empty else None

    fin = financed_by_year[
        (financed_by_year["bank_id"] == bank_id) &
        (financed_by_year["reporting_year"] == 2024)
    ]
    financed    = float(fin["financed_em_loans_tco2e"].values[0])                  if not fin.empty else None
    intensity   = float(fin["carbon_intensity_tco2e_per_meur_lending"].values[0]) if not fin.empty else None

    fs_2024 = fs[(fs["bank_id"] == bank_id) & (fs["reporting_year"] == 2024)]
    def fs_val(col):
        if fs_2024.empty or col not in fs_2024.columns:
            return None
        v = fs_2024[col].values[0]
        return None if pd.isna(v) else round(float(v), 4)

    gov = tables["governance"][
        (tables["governance"]["bank_id"] == bank_id) &
        (tables["governance"]["reporting_year"] == 2024)
    ]
    gov_maturity = None
    if not gov.empty:
        g = gov.iloc[0]
        gov_maturity = {
            "esg_committee_exists":        bool(g.get("esg_committee_exists")),
            "esg_committee_meetings":      int(g.get("esg_committee_meetings_per_year", 0)),
            "board_climate_expertise_pct": float(g.get("board_climate_expertise_pct", 0)),
            "ceo_esg_compensation_pct":    float(g.get("ceo_esg_compensation_pct", 0)),
            "external_assurance":          g.get("external_assurance"),
            "erm_integration":             bool(g.get("erm_integration_flag")),
        }

    tgt = targets[targets["bank_id"] == bank_id]
    target_summary = [
        {
            "target_id":              row["target_id"],
            "type":                   row["target_type"],
            "scope":                  row["scope_coverage"],
            "status":                 row["status"],
            "target_year":            int(row["target_year"]),
            "framework":              row["target_framework"],
            "target_progress_pct_2024":    row.get("target_progress_pct_2024"),   # real achievement only; None if unknown
            "schedule_elapsed_pct_2024":   row.get("schedule_elapsed_pct_2024"),  # time fraction of target window elapsed by 2024
            "progress_basis":              row.get("progress_basis"),
            "progress_is_schedule_proxy":  row.get("progress_is_schedule_proxy"),
        }
        for _, row in tgt.iterrows()
    ]

    sov_bank = sovereign_bonds[sovereign_bonds["bank_id"] == bank_id]
    sov_gaps = int(sov_bank["data_gap_flag"].sum()) if not sov_bank.empty else 0

    # ← NEW: carbon credit quality summary
    cc = tables["carbon_credits"]
    cc_bank = cc[(cc["bank_id"] == bank_id) & (cc["reporting_year"] == 2024)]
    cc_summary = None
    if not cc_bank.empty:
        retired = cc_bank[cc_bank["use"] == "retirement_against_emissions"]
        cc_summary = {
            "total_credits_tco2e":       int(cc_bank["tonnes_co2e"].sum()),
            "retired_tco2e":             int(retired["tonnes_co2e"].sum()) if not retired.empty else 0,
            "removal_pct":               round(
                cc_bank[cc_bank["credit_type"] == "carbon_removal"]["tonnes_co2e"].sum() /
                cc_bank["tonnes_co2e"].sum() * 100, 1
            ) if cc_bank["tonnes_co2e"].sum() > 0 else 0,
            "high_permanence_pct":       round(
                cc_bank[cc_bank["permanence_rating"] == "high"]["tonnes_co2e"].sum() /
                cc_bank["tonnes_co2e"].sum() * 100, 1
            ) if cc_bank["tonnes_co2e"].sum() > 0 else 0,
            "additionality_verified_pct": round(
                cc_bank[cc_bank["additionality_verified"] == True]["tonnes_co2e"].sum() /
                cc_bank["tonnes_co2e"].sum() * 100, 1
            ) if cc_bank["tonnes_co2e"].sum() > 0 else 0,
        }

    return {
        # GHG emissions
        "scope1_2024_tco2e":                    scope1_total,
        "scope2_location_2024_tco2e":           scope2_location,
        "scope2_market_2024_tco2e":             scope2_market,
        "scope3_travel_2024_tco2e":             scope3,
        "financed_emissions_2024_tco2e":        financed,
        "carbon_intensity_2024_tco2e_per_meur": intensity,

        # Financial context
        "green_loans_pct_2024":     fs_val("green_loans_pct"),
        "climate_capex_2024_meur":  fs_val("climate_capex_meur"),
        "climate_opex_2024_meur":   fs_val("climate_opex_meur"),
        "total_assets_2024_meur":   fs_val("total_assets_meur"),
        "total_loans_2024_meur":    fs_val("total_loans_meur"),

        # Governance + targets
        "governance_maturity": gov_maturity,
        "target_summary":      target_summary,

        # ← NEW: sector exposure KPIs
        "high_carbon_sector_exposure_pct": sector_kpis[bank_id]["high_carbon_sector_exposure_pct"],
        "fossil_fuel_exposure_pct":        sector_kpis[bank_id]["fossil_fuel_exposure_pct"],
        "high_carbon_sector_exposure_meur": sector_kpis[bank_id]["high_carbon_sector_exposure_meur"],
        "fossil_fuel_exposure_meur":        sector_kpis[bank_id]["fossil_fuel_exposure_meur"],

        # ← NEW: data quality summary
        "emissions_data_quality_summary": dq_summaries[bank_id],

        # ← NEW: carbon credit quality summary
        "carbon_credit_summary": cc_summary,

        # Data quality flags
        "scope1_fleet_included":                True,
        "scope3_travel_comparative_available":  False,
        "listed_equity_emissions_are_proxy":    True,
        "sovereign_bonds_with_data_gaps":       sov_gaps,
    }


# ── 8. SECTION-SPECIFIC PAYLOAD SLICER ────────────────────────
# ← NEW FIX: pre-slice payloads per section to avoid token bloat
# and prevent cross-section context bleed in the Writer Agent.

SECTION_KEYS = {
    "general_requirements": [
        "metadata", "bank", "financial_summary", "general_requirements_context",
        "targets", "scope1", "scope2", "scope3_travel", "financed_emissions",
    ],
    "governance": [
        "metadata", "bank", "governance", "board_minutes", "climate_risk_register",
    ],
    "strategy": [
        "metadata", "bank", "financial_summary", "climate_scenarios",
        "climate_risk_register", "value_chain_map", "climate_opportunities", "targets",
        "transition_plan", "resilience_assessment", "climate_financial_effects",
    ],
    "risk_management": [
        "metadata", "bank", "climate_risk_register", "physical_risk_exposures",
        "value_chain_map", "governance", "climate_financial_effects",
    ],
    "metrics_targets": [
        "metadata", "bank", "financial_summary", "scope1", "scope2",
        "scope3_travel", "financed_emissions", "financed_emissions_equity",
        "financed_emissions_sovereign", "targets", "carbon_credits",
        "internal_carbon_price",
        "scope3_categories", "ghg_methodology", "scope12_consolidation",
    ],
}
# reporting_kpis is always injected into every section payload
# executive_summary is assembled last from all sections — no pre-slice needed


# ── 9. BUILD MASTER REPORTING PAYLOAD ────────────────────────
reporting_payload = {}

for bank_id in tables["banks"]["bank_id"].unique():

    def bank_slice(df):
        """Return records for the active bank when a table is bank-scoped.

        Some helper/reference tables, such as source_systems, are global and do
        not contain a bank_id column. In that case, keep the full table instead
        of crashing with KeyError: 'bank_id'.
        """
        if df is None:
            return []
        if not hasattr(df, "columns"):
            return []
        if getattr(df, "empty", False):
            return []
        if "bank_id" not in df.columns:
            return df.to_dict("records")
        return df[df["bank_id"] == bank_id].to_dict("records")

    def df_slice(df):
        return bank_slice(df)

    def bank_slice_opt(name):   # new disclosure tables may be absent if not generated yet
        return bank_slice(tables[name]) if name in tables else []

    # ── Metadata ─────────────────────────────────────────────
    metadata = {
        "bank_id": bank_id,
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],

        "data_gaps": [
            {
                "field": "scope1_fleet_tco2e",
                "affected_years": [2022, 2023],
                "reason": "Vehicle activity data available for 2024 only.",
                "instruction": "Report 2024 fleet Scope 1 only. State data unavailable for prior years. Do not report missing years as zero."
            },
            {
                "field": "scope3_travel_tco2e",
                "affected_years": [2022, 2023],
                "reason": "Travel records cover 2024 only.",
                "instruction": "Report 2024 Scope 3 Cat.6 only. Prior years not available."
            },
            {
                "field": "investments.counterparty_id",
                "affected_years": [2024],
                "reason": "counterparty_id not populated in investments table. Investments attributed at bank level only.",
                "instruction": "Note PCAF DQS 3 for investment attribution. Do not report issuer-level investment claims."
            },
            # ← NEW: static loan book caveat
            {
                "field": "total_loans_meur",
                "affected_years": [2022, 2023, 2024],
                "reason": "total_loans_meur is constant across all reporting years in the available data. Loan book growth data not available.",
                "instruction": "Carbon intensity trend reflects emission changes only, not loan book growth. Disclose this methodology limitation explicitly. Do not describe a trend in the loan book denominator."
            },
        ],

        "pcaf_methodology": {
            "corporate_loans": "outstanding_amount_meur / evic_meur * total_ghg_tco2e (PCAF Standard §B62)",
            "sovereign_bonds_investments": "nominal_amount_meur / ppp_gdp_meur * national_scope1_tco2e (PCAF Standard §B62A)",
            "listed_equity": (
                "market_value_meur / issuer_evic_meur * issuer_revenue_meur "
                "(PCAF Standard §B61 proxy — revenue substituted for direct issuer emissions "
                "due to absence of counterparty-level emission data in investment records)"
            ),
            "sovereign_bonds_note": (
                "issuer_evic_meur and issuer_revenue_meur are null for sovereign bonds by design. "
                "EVIC and revenue are corporate concepts not applicable to sovereigns."
            ),
            "investments_counterparty_note": (
                "counterparty_id is null across all 185 investment rows. "
                "This is a known data generation gap. Attribution is at bank level only."
            ),
        },

        # ← NEW: Scope 2 REC reconciliation note
        "scope2_rec_reconciliation": (
            "scope2_market_tco2e is sourced directly from utility_invoices.scope2_market_tco2e, "
            "which already reflects market-based accounting with grid emission factors adjusted "
            "for renewable energy certificates and PPAs. The rec_registry.csv data is not "
            "subtracted again during preparation — it is used for disclosure narrative only "
            "(certificate type, volume, registry). There is no double-counting."
        ),

        # ← NEW: risk_rating derivation formula
        "risk_rating_methodology": (
            "climate_risk_register.risk_rating is derived from a 5x5 risk matrix: "
            "scores 1-2 = low, 3-6 = medium, 8-12 = high, 15-25 = critical. "
            "Specifically: likelihood_score * severity_score; "
            "critical >= 15, high >= 8, medium >= 3, low < 3. "
            "This formula must be used verbatim if the rating methodology is described in the report."
        ),

        "vehicles_correction": {
            "method": "Scope 1 fleet recalculated from annual_fuel_consumption_l * 2.67 kg_CO2/litre / 1000",
            "reason": "Pre-computed scope1_tco2e_2024 had errors including non-zero values for electric vehicles.",
            "electric_vehicles_zeroed": True,
            "emission_factor_source": "Empirically derived — consistent 2.67 kg CO2/litre across all fuel types",
        },
    }

    # ── General requirements context ────────────────────────────
    _bank_row = tables["banks"][tables["banks"]["bank_id"] == bank_id].to_dict("records")[0]
    _gov_2024 = tables["governance"][(tables["governance"]["bank_id"] == bank_id) & (tables["governance"]["reporting_year"] == 2024)]
    _gov_row = _gov_2024.to_dict("records")[0] if not _gov_2024.empty else {}
    general_requirements_context = {
        "reporting_entity": _bank_row.get("bank_name"),
        "bank_id": bank_id,
        "reporting_period_end": _bank_row.get("fiscal_year_end"),
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],
        "reporting_currency": _bank_row.get("reporting_currency"),
        "boundary_type": _bank_row.get("boundary_type"),
        "regulatory_regime": _bank_row.get("regulatory_regime"),
        "standards_basis": "IFRS S1 / IFRS S2-style sustainability-related financial disclosure basis",
        "source_systems": bank_slice(tables["source_systems"]) if "source_systems" in tables else [],
        "data_gaps": metadata["data_gaps"],
        "pcaf_methodology": metadata["pcaf_methodology"],
        "scope2_rec_reconciliation": metadata["scope2_rec_reconciliation"],
        "risk_rating_methodology": metadata["risk_rating_methodology"],
        "external_assurance": _gov_row.get("external_assurance"),
        "assurance_provider": _gov_row.get("assurance_provider"),
        "assurance_scope": _gov_row.get("assurance_scope"),
        "assurance_standard": _gov_row.get("assurance_standard"),
        "coherence_fixes_applied": [f for f in coherence_fixes_applied if f.get("bank_id") == bank_id],
    }

    # ── Full payload assembly ─────────────────────────────────
    full_payload = {
        "metadata":                     metadata,
        "general_requirements_context": general_requirements_context,
        "reporting_kpis":       build_reporting_kpis(bank_id),
        "bank":                         tables["banks"][tables["banks"]["bank_id"] == bank_id].to_dict("records")[0],
        "financial_summary":            bank_slice(fs),
        "scope1":                       bank_slice(scope1),
        "scope2":                       bank_slice(scope2),
        "scope3_travel":                bank_slice(scope3_travel),
        "financed_emissions":           bank_slice(financed_by_year),
        "financed_emissions_equity":    bank_slice(listed_equity),
        "financed_emissions_sovereign": bank_slice(sovereign_bonds),
        "targets":                      bank_slice(targets),
        "governance":                   bank_slice(tables["governance"]),
        "board_minutes":                bank_slice(tables["board_minutes_extract"]),
        "climate_scenarios":            bank_slice(tables["climate_scenarios"]),
        "climate_risk_register":        bank_slice(tables["climate_risk_register"]),
        "physical_risk_exposures":      bank_slice(tables["physical_risk_exposures"]),
        "carbon_credits":               bank_slice(tables["carbon_credits"]),
        "internal_carbon_price":        bank_slice(tables["internal_carbon_price"]),
        "value_chain_map":              bank_slice(tables["value_chain_map"]),
        "climate_opportunities":        bank_slice(tables["climate_opportunities"]),  # ← NEW
        # ── Added IFRS S1/S2 disclosure tables (present only if generated) ──
        "scope3_categories":            bank_slice_opt("scope3_categories"),
        "transition_plan":              bank_slice_opt("transition_plan"),
        "climate_financial_effects":    bank_slice_opt("climate_financial_effects"),
        "resilience_assessment":        bank_slice_opt("resilience_assessment"),
        "ghg_methodology":              bank_slice_opt("ghg_methodology"),
        "scope12_consolidation":        bank_slice_opt("scope12_consolidation"),
    }

    reporting_payload[bank_id] = full_payload


# ── 10. BUILD SECTION-SPECIFIC PAYLOADS ──────────────────────
# ← NEW FIX: one payload per section per bank
section_payloads = {}

for bank_id, full_payload in reporting_payload.items():
    section_payloads[bank_id] = {}
    kpis = full_payload["reporting_kpis"]

    for section_name, keys in SECTION_KEYS.items():
        section_payload = {k: full_payload[k] for k in keys if k in full_payload}
        # Always inject the KPI block — every section needs it
        section_payload["reporting_kpis"] = kpis
        section_payloads[bank_id][section_name] = section_payload

    print(f"{bank_id} section payloads: "
          + ", ".join(f"{s}={len(json.dumps(p, default=str))} chars"
                      for s, p in section_payloads[bank_id].items()))



# ── JSON CLEANING HELPERS ─────────────────────────────────────
def clean_for_json(value):
    """
    Recursively convert pandas/numpy missing values to JSON null.

    This function must be defined before export because json.dump(...,
    allow_nan=False) will fail on NaN/Infinity values.
    """
    if isinstance(value, dict):
        return {k: clean_for_json(v) for k, v in value.items()}

    if isinstance(value, list):
        return [clean_for_json(v) for v in value]

    # Convert pandas/numpy scalar values to native Python values when possible.
    if hasattr(value, "item") and not isinstance(value, (str, bytes)):
        try:
            value = value.item()
        except Exception:
            pass

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return value


def dump_strict_json(obj, path: Path):
    """
    Write standards-compliant JSON:
    - missing values become null;
    - NaN/Infinity are rejected;
    - UTF-8 is preserved.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            clean_for_json(obj),
            f,
            indent=2,
            default=str,
            allow_nan=False,
            ensure_ascii=False,
        )


# ── 11. EXPORT ────────────────────────────────────────────────
OUTPUT_PATH.mkdir(exist_ok=True)

# Full payloads
for bank_id, payload in reporting_payload.items():
    dump_strict_json(payload, OUTPUT_PATH / f"payload_{bank_id}.json")

# Section-specific payloads
for bank_id, sections in section_payloads.items():
    for section_name, section_payload in sections.items():
        fname = OUTPUT_PATH / f"payload_{bank_id}_{section_name}.json"
        dump_strict_json(section_payload, fname)

print(f"\nExported {len(reporting_payload)} full payloads")
print(f"Exported {len(reporting_payload) * len(SECTION_KEYS)} section payloads")
print("All files in payloads/ directory")
print("\nFull payload keys:", list(reporting_payload["BANK01"].keys()))
print("Section payload keys per bank:", list(section_payloads["BANK01"].keys()))


# ── 12. EXPORT VALIDATION ────────────────────────────────────
import re

def validate_exported_payloads(output_path: Path, expected_banks: int):
    """
    Validate that exported payloads are strict JSON and contain the
    section-specific evidence needed by the report-generation agents.
    """
    payload_files = sorted(output_path.glob("payload_BANK*.json"))
    if not payload_files:
        raise FileNotFoundError(f"No payload files found in {output_path}")

    invalid_tokens = re.compile(r'(?<!")\b(?:NaN|Infinity|-Infinity)\b(?!")')

    for file_path in payload_files:
        text = file_path.read_text(encoding="utf-8")

        if invalid_tokens.search(text):
            raise ValueError(f"Invalid JSON numeric token found in {file_path.name}")

        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{file_path.name} is not valid JSON: {exc}") from exc

        if file_path.name.endswith("_governance.json"):
            if "climate_risk_register" not in data:
                raise ValueError(f"{file_path.name} missing climate_risk_register")
            if len(data.get("climate_risk_register") or []) == 0:
                raise ValueError(f"{file_path.name} has empty climate_risk_register")

        if file_path.name.endswith("_strategy.json"):
            if "targets" not in data:
                raise ValueError(f"{file_path.name} missing targets")
            if len(data.get("targets") or []) == 0:
                raise ValueError(f"{file_path.name} has empty targets")

        if file_path.name.endswith("_general_requirements.json"):
            if "general_requirements_context" not in data:
                raise ValueError(f"{file_path.name} missing general_requirements_context")
            if not data.get("general_requirements_context"):
                raise ValueError(f"{file_path.name} has empty general_requirements_context")

    full_payload_count = len([
        p for p in payload_files
        if p.name.count("_") == 1 and p.name.endswith(".json")
    ])
    section_payload_count = len(payload_files) - full_payload_count

    if full_payload_count != expected_banks:
        raise ValueError(
            f"Expected {expected_banks} full payloads, found {full_payload_count}"
        )

    print("\nValidation passed")
    print(f"- Strict JSON files checked: {len(payload_files)}")
    print(f"- Full payloads: {full_payload_count}")
    print(f"- Section payloads: {section_payload_count}")
    print("- No NaN, Infinity, or -Infinity numeric tokens found")
    print("- Governance payloads include climate_risk_register")
    print("- Strategy payloads include targets")


validate_exported_payloads(
    OUTPUT_PATH,
    expected_banks=len(reporting_payload),
)



Loaded 33 tables
  disclosures: 43 rows
  climate_opportunities: 25 rows
  climate_scenarios: 84 rows (has methodology_notes: True)
Targets: milestones parsed. Progress computed honestly (schedule vs actual separated).
   target_id bank_id  target_year  schedule_elapsed_pct_2024 actual_progress_pct_2024         progress_basis
0     TGT001  BANK01         2030                       40.0                     None  schedule_elapsed_only
1     TGT002  BANK01         2030                       25.0                     None  schedule_elapsed_only
2     TGT003  BANK01         2050                       13.3                     None  schedule_elapsed_only
3     TGT004  BANK02         2030                       40.0                     None  schedule_elapsed_only
4     TGT005  BANK02         2030                       25.0                     None  schedule_elapsed_only
5     TGT006  BANK02         2050                       13.3                     None  schedule_elapsed_only
6     TGT007  BANK

In [18]:
# === 2.8 SOURCE-DATA COHERENCE VALIDATION (integrity gate) ==================
# Catches cross-field incoherence that the generation pipeline's judge cannot see:
# computed metrics vs passed-through target/scenario/governance figures. It does NOT
# rewrite management-asserted records — it flags exactly which raw CSV rows to fix.
# Schema-agnostic: it discovers columns by pattern so it works on the real CSV schema.
import re as _re

STRICT_COHERENCE = False  # set True to raise on any 'error'-severity finding

def _cnum(v):
    try:
        f = float(v)
        return f if f == f else None  # drop NaN
    except Exception:
        return None

def _find_col(df, *pats):
    for p in pats:
        for c in df.columns:
            if _re.search(p, str(c), _re.I):
                return c
    return None

def run_data_coherence_validation(write=True):
    report = {}
    bank_ids = list(tables["banks"]["bank_id"])

    cs_all  = tables.get("climate_scenarios")
    gov_all = tables.get("governance")

    for bid in bank_ids:
        findings = []
        def add(sev, check, detail, **kw):
            findings.append({"severity": sev, "check": check, "detail": detail, **kw})

        # ---- context (computed, trustworthy) ----
        brow = tables["banks"][tables["banks"]["bank_id"] == bid]
        loans = _cnum(brow["total_loans_meur"].values[0]) if not brow.empty and "total_loans_meur" in brow else None
        fr = financed_by_year[(financed_by_year["bank_id"] == bid) &
                              (financed_by_year["reporting_year"] == 2024)]
        fe    = _cnum(fr["financed_em_loans_tco2e"].values[0]) if not fr.empty else None
        inten = _cnum(fr["carbon_intensity_tco2e_per_meur_lending"].values[0]) if not fr.empty else None

        # 1) financed-emissions intensity plausibility band
        if inten is not None and not (1.0 <= inten <= 2000.0):
            add("warn", "financed_intensity_band",
                f"Computed financed-emissions intensity {inten:.2f} tCO2e/EURm is outside the 1-2000 "
                f"plausibility band; check counterparty emissions / exposures in the source.",
                value=inten)

        # 2) target baselines vs computed metrics
        tgt = targets[targets["bank_id"] == bid]
        bcol = _find_col(tgt, "baseline_value", "baseline.*value")
        scol = _find_col(tgt, "scope_coverage", "scope")
        ucol = _find_col(tgt, "metric_unit", "unit", "target_metric", "metric")
        tcol = _find_col(tgt, "target_type", "type")
        rows = []
        for _, r in tgt.iterrows():
            bv    = _cnum(r.get(bcol)) if bcol else None
            scope = str(r.get(scol, "")).lower() if scol else ""
            unit  = str(r.get(ucol, "")).lower() if ucol else ""
            ttype = str(r.get(tcol, "")).lower() if tcol else ""
            tid   = r.get("target_id")
            rows.append((tid, bv, scope, ttype))

            is_intensity = ("inten" in ttype) or ("per_meur" in unit) or ("per eur" in unit) or ("intensity" in unit)
            if is_intensity and bv and inten and bv > 0:
                ratio = inten / bv
                if ratio > 10 or ratio < 0.1:
                    add("error", "intensity_target_scale_mismatch",
                        f"Target {tid} baseline {bv} (unit '{unit or '?'}') vs computed 2024 intensity "
                        f"{inten:.2f} differ ~{ratio:.0f}x - same metric cannot hold both. Fix the "
                        f"baseline scale/units in targets.csv (or the financed-emissions source).",
                        target_id=tid, baseline=bv, computed_intensity=round(inten, 2), ratio=round(ratio, 1))

            all_scopes = ("all" in scope and "scope" in scope) or ("scope 3" in scope) \
                         or ("financed" in scope) or ("net" in ttype and "zero" in ttype)
            # absolute-emissions baselines only: an INTENSITY baseline vs an absolute
            # total would mix units (scale-mismatch already covers intensity targets).
            if all_scopes and not is_intensity and bv is not None and fe is not None and bv < fe:
                add("error", "baseline_below_financed_emissions",
                    f"Target {tid} ({scope or 'broad scope'}) baseline {bv:.0f} tCO2e is below 2024 financed "
                    f"emissions of {fe:.0f} tCO2e. A financed-inclusive/all-scopes baseline cannot be smaller "
                    f"than financed emissions alone.",
                    target_id=tid, baseline=bv, financed_emissions=round(fe, 0))

        # 2c) broad-scope baseline below a narrower (Scope 1&2) baseline
        narrow = [bv for _, bv, sc, _ in rows if bv is not None and ("1" in sc and "2" in sc and "3" not in sc)]
        broad  = [(tid, bv, sc) for tid, bv, sc, tt in rows
                  if bv is not None and (("all" in sc) or ("net" in tt and "zero" in tt))]
        if narrow and broad:
            nmax = max(narrow)
            for tid, bv, sc in broad:
                if bv < nmax:
                    add("error", "baseline_scope_non_monotonic",
                        f"Broad-scope target {tid} baseline {bv:.0f} is below a narrower Scope 1&2 baseline "
                        f"{nmax:.0f}. A broader scope must carry a >= baseline.",
                        target_id=tid, broad_baseline=bv, narrow_baseline=nmax)

        # 3) scenario magnitudes vs exposure
        if cs_all is not None and "bank_id" in cs_all.columns:
            cs = cs_all[cs_all["bank_id"] == bid]
            sc_name = _find_col(cs, "scenario_name", "scenario", "name")
            euro_cols = [c for c in cs.columns
                         if _re.search(r"meur", str(c), _re.I)
                         and _re.search(r"stranded|revenue_at_risk|loss|exposure_at_risk", str(c), _re.I)]
            for c in euro_cols:
                for _, r in cs.iterrows():
                    v = _cnum(r.get(c))
                    if v is not None and loans and v > loans:
                        add("warn", "scenario_euro_exceeds_loans",
                            f"Scenario '{c}'={v:.0f} EURm exceeds total loans {loans:.0f} EURm - implausible "
                            f"absolute magnitude; check units/scope in climate_scenarios.csv.",
                            column=c, value=v, total_loans=loans,
                            scenario=str(r.get(sc_name, "")) if sc_name else None)
            pct_cols = [c for c in cs.columns if _re.search(r"pct_capital", str(c), _re.I)]
            for c in pct_cols:
                for _, r in cs.iterrows():
                    v = _cnum(r.get(c))
                    if v is not None and (v < 0 or v > 100):
                        add("warn", "scenario_pct_capital_out_of_range",
                            f"Scenario '{c}'={v} is outside 0-100% of capital.", column=c, value=v)

        # 4) board percentages must resolve to whole directors
        if gov_all is not None and "bank_id" in gov_all.columns:
            gov = gov_all[gov_all["bank_id"] == bid]
            if "reporting_year" in gov.columns:
                gov = gov[gov["reporting_year"] == 2024]
            if not gov.empty:
                g = gov.iloc[0]
                bs_col = _find_col(gov, "board_size", "num_board", "board_members", "n_board")
                bs = _cnum(g.get(bs_col)) if bs_col else None
                if bs and bs > 0:
                    pcols = [c for c in gov.columns
                             if _re.search(r"(independent|expertise|climate).*pct|pct.*(independent|expertise)",
                                           str(c), _re.I)]
                    for pcol in pcols:
                        pv = _cnum(g.get(pcol))
                        if pv is not None:
                            cnt = pv / 100.0 * bs
                            if abs(cnt - round(cnt)) > 0.05:
                                add("warn", "board_pct_not_integer",
                                    f"governance '{pcol}'={pv}% of board_size {int(bs)} = {cnt:.2f} directors "
                                    f"(not whole). Nearest coherent value: {round(round(cnt)/bs*100,1)}%.",
                                    column=pcol, pct=pv, board_size=int(bs),
                                    implied_count=round(cnt, 2), suggested_pct=round(round(cnt)/bs*100, 1))

        report[bid] = {
            "findings": findings,
            "error_count": sum(1 for f in findings if f["severity"] == "error"),
            "warn_count":  sum(1 for f in findings if f["severity"] == "warn"),
        }

    total_err  = sum(r["error_count"] for r in report.values())
    total_warn = sum(r["warn_count"] for r in report.values())
    print(f"=== DATA COHERENCE VALIDATION ===  errors={total_err}  warnings={total_warn}")
    for bid, r in report.items():
        if r["findings"]:
            print(f"\n[{bid}] errors={r['error_count']} warnings={r['warn_count']}")
            for f in r["findings"]:
                print(f"  {f['severity'].upper():5s} {f['check']}: {f['detail']}")
    if write:
        from pathlib import Path as _P
        _P("data_coherence_report.json").write_text(json.dumps(report, indent=2, default=str))
        print("\nWrote data_coherence_report.json")
    if STRICT_COHERENCE and total_err:
        raise RuntimeError(f"Data coherence validation failed: {total_err} error-severity finding(s). "
                           f"Fix the source CSVs or set STRICT_COHERENCE=False.")
    return report

coherence_report = run_data_coherence_validation()


=== DATA COHERENCE VALIDATION ===  errors=1  warnings=7

[BANK01] errors=0 warnings=1
  WARN  board_pct_not_integer: governance 'all_exec_climate_remuneration_pct'=15.1% of board_size 10 = 1.51 directors (not whole). Nearest coherent value: 20.0%.

[BANK02] errors=0 warnings=2
  WARN  board_pct_not_integer: governance 'all_exec_climate_remuneration_pct'=14.1% of board_size 9 = 1.27 directors (not whole). Nearest coherent value: 11.1%.
  WARN  board_pct_not_integer: governance 'climate_on_board_agenda_pct'=75.0% of board_size 9 = 6.75 directors (not whole). Nearest coherent value: 77.8%.

[BANK03] errors=0 warnings=1
  WARN  board_pct_not_integer: governance 'all_exec_climate_remuneration_pct'=14.9% of board_size 10 = 1.49 directors (not whole). Nearest coherent value: 10.0%.

[BANK04] errors=0 warnings=1
  WARN  board_pct_not_integer: governance 'all_exec_climate_remuneration_pct'=23.6% of board_size 12 = 2.83 directors (not whole). Nearest coherent value: 25.0%.

[BANK05] errors=1 war

### 2.9 Final fixes added

This notebook version includes dynamic coherence fixes before payload export:

- adds `general_requirements` section payload export;
- writes strict JSON with `null` instead of `NaN`;
- fixes financed-emissions target scale using calculated carbon intensity;
- fixes broad all-scopes/net-zero baselines that were below financed emissions;
- aligns bank master financial fields with 2024 `financial_summary`;
- rounds board percentages to whole director counts and recalculates board climate agenda percentage from board minutes;
- recalculates/caps scenario absolute monetary magnitudes;
- aligns obvious risk mitigation actions with risk type;
- records all automated coherence fixes in `general_requirements_context.coherence_fixes_applied`.
